## Load dependencies

In [ ]:
try:
    import kagglehub
except (ImportError, ModuleNotFoundError):
    print("kagglehub not installed or not running in Kaggle; continuing.")


In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
except (ImportError, ModuleNotFoundError):
    print("Not running in Google Colab (or google.colab not available); skipping Drive mount.")


In [2]:
import importlib
import subprocess
import sys


def _ensure(pkg, import_name=None):
    """Import `import_name` (default: `pkg`), installing `pkg` via pip if needed.

    Tries a normal `pip install` first, then falls back to
    `--break-system-packages` for externally-managed Python environments
    (common on recent Debian/Ubuntu images).
    """
    import_name = import_name or pkg
    try:
        importlib.import_module(import_name)
        return
    except ImportError:
        pass

    last_err = None
    for extra_args in ([], ["--break-system-packages"]):
        try:
            subprocess.run(
                [sys.executable, "-m", "pip", "install", "-q", *extra_args, pkg],
                check=True,
            )
            importlib.import_module(import_name)
            return
        except (subprocess.CalledProcessError, ImportError) as e:
            # Catch only the specific failure modes we can actually recover from
            # (install failing, or the module still not importable afterward) --
            # not bare Exception, which was also swallowing KeyboardInterrupt and
            # discarding the real error message (network down, disk full, bad
            # package name, permission denied), leaving only an uninformative
            # "Could not install" with no clue why.
            last_err = e
            continue

    raise RuntimeError(
        f"Could not install '{pkg}': {last_err}\n"
        f"Please install it manually, e.g.:\n"
        f"    pip install {pkg}"
    ) from last_err


for _pkg, _mod in [("torch", "torch"), ("numpy", "numpy"), ("safetensors", "safetensors")]:
    _ensure(_pkg, _mod)

print("dependencies OK")


dependencies OK


## Write `tensegrity_lm.py`

In [3]:
%%writefile tensegrity_lm.py
"""
TensegrityLM — a byte-level decoder-only language model.

Single-file, checkpointable, trainable implementation combining:
  - Rotary positional embeddings (RoPE)              [Geometry]
  - RMSNorm pre-norm transformer blocks              [Algebraic Math]
  - Grouped-Query Attention (GQA)                    [Algorithms]
  - Alternating local sliding-window / global causal attention [Space]
  - SwiGLU feed-forward                              [Social Dynamics]
  - Learnable per-layer residual gain + activation-RMS
    "homeostatic" set-point regularization           [Biology]
  - Depth-scaled stochastic layer drop (LayerDrop)   [Evolution]
  - Cosine LR schedule with warmup ("temperature annealing") [Physics]
  - Optional confidence head for adaptive sampling temperature
                                                      [Psychology]
  - Byte-level tokenizer, zero external deps         [English Language]

Honesty note: every individual mechanism above is established in the
literature (RoPE, GQA, SwiGLU, RMSNorm, LayerDrop,
LayerScale-style gains, cosine schedules). The specific combination —
particularly the homeostatic residual-gain regularization framed as an
explicit activation-RMS set-point loss, paired with depth-scaled layer
drop and alternating local/global GQA — is a coherent but UNVALIDATED
recombination. Train it, measure it, don't assume it beats a vanilla
transformer of equal size until you've run the comparison yourself.

Usage:
    python tensegrity_lm.py prepare --data_path mydata.txt --out_dir data/mycorpus
    python tensegrity_lm.py train    --data_dir data/mycorpus --out_dir runs/run1
    python tensegrity_lm.py train    --data_dir data/mycorpus --out_dir runs/run1 --init_from resume
    python tensegrity_lm.py generate --ckpt runs/run1/ckpt_best.pt --prompt "Hello" --max_new_tokens 200
    python tensegrity_lm.py eval     --ckpt runs/run1/ckpt_best.pt --data_dir data/mycorpus
"""

import argparse
import dataclasses
import glob
import json
import math
import os
import sys
import time
import inspect
import random
from dataclasses import dataclass, asdict
from typing import Optional, Tuple, List, Dict, Any

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.distributed as dist

try:
    _SDPA_SUPPORTS_GQA = "enable_gqa" in inspect.signature(F.scaled_dot_product_attention).parameters
except (TypeError, ValueError):
    # Some torch builds expose scaled_dot_product_attention as a builtin with no
    # introspectable signature at all (inspect.signature raises ValueError), which
    # previously crashed this entire module at import time. Fall back to the
    # pre-GQA code path instead of hard-failing on those builds.
    _SDPA_SUPPORTS_GQA = False

# =============================================================================
# Tokenizer
# =============================================================================

class ByteTokenizer:
    """
    Byte-level tokenizer. Vocabulary = 256 raw byte values + 3 special tokens:
        256 = BOS (beginning of sequence)
        257 = EOS (end of sequence / document separator)
        258 = PAD (padding, unused by default but reserved)

    Zero training required.

    Losslessness note:
      - `encode_bytes`/`decode_bytes` are lossless for ANY bytes input.
      - `encode`/`decode` round-trip losslessly for any *well-formed* Python
        str, because UTF-8 encode/decode of valid unicode is bijective.
        They are NOT lossless if `errors` is set to a lossy mode (the
        default, "replace", is lossy and matches prior behavior for
        backward compatibility) and the input contains content that
        can't be represented in strict UTF-8 (e.g. lone surrogates).
        Pass errors="strict" if you need a hard failure instead of silent
        corruption, or use encode_bytes for guaranteed lossless behavior.
    """

    VOCAB_SIZE = 259
    BOS = 256
    EOS = 257
    PAD = 258
    _SPECIAL_IDS = frozenset((BOS, EOS, PAD))
    _SPECIAL_TOKEN_STRINGS = {
        BOS: "<|bos|>",
        EOS: "<|eos|>",
        PAD: "<|pad|>",
    }

    def __len__(self) -> int:
        return self.VOCAB_SIZE

    # ------------------------------------------------------------------ #
    # str <-> ids
    # ------------------------------------------------------------------ #
    def encode(
        self,
        text: str,
        errors: str = "replace",
        add_bos: bool = False,
        add_eos: bool = False,
    ) -> List[int]:
        """
        errors: passed through to str.encode. Default "replace" preserves
            prior behavior (never raises). Use "strict" to detect
            unencodable input instead of silently corrupting it.
        add_bos/add_eos: default False, preserves prior output shape.
        """
        ids = list(text.encode("utf-8", errors=errors))
        if add_bos:
            ids.insert(0, self.BOS)
        if add_eos:
            ids.append(self.EOS)
        return ids

    def decode(
        self,
        ids: Any,
        errors: str = "replace",
        skip_special_tokens: bool = True,
        strict: bool = False,
    ) -> str:
        """
        skip_special_tokens: if True (default), silently drops ids outside 0-255.
            If False, formats special token IDs (BOS, EOS, PAD) into readable strings (<|bos|>, etc.).
        strict: if True, raise ValueError on any id outside [0, VOCAB_SIZE)
            rather than silently dropping it. Off by default to preserve
            prior (permissive) behavior.
        """
        if hasattr(ids, "tolist"):
            ids = ids.tolist()
        elif not isinstance(ids, (list, tuple)):
            ids = list(ids)

        if strict:
            bad = [int(i) for i in ids if not (0 <= int(i) < self.VOCAB_SIZE)]
            if bad:
                raise ValueError(f"decode: invalid token id(s) outside vocab: {bad[:5]}"
                                  f"{'...' if len(bad) > 5 else ''}")

        if skip_special_tokens:
            byte_vals = bytes(int(i) for i in ids if 0 <= int(i) <= 255)
            return byte_vals.decode("utf-8", errors=errors)

        parts = []
        cur_bytes = bytearray()
        for raw_id in ids:
            i = int(raw_id)
            if 0 <= i <= 255:
                cur_bytes.append(i)
            elif i in self._SPECIAL_TOKEN_STRINGS:
                if cur_bytes:
                    parts.append(cur_bytes.decode("utf-8", errors=errors))
                    cur_bytes.clear()
                parts.append(self._SPECIAL_TOKEN_STRINGS[i])
        if cur_bytes:
            parts.append(cur_bytes.decode("utf-8", errors=errors))
        return "".join(parts)

    # ------------------------------------------------------------------ #
    # bytes <-> ids (the actually-lossless path)
    # ------------------------------------------------------------------ #
    def encode_bytes(
        self,
        data: bytes,
        add_bos: bool = False,
        add_eos: bool = False,
    ) -> List[int]:
        if not isinstance(data, (bytes, bytearray)):
            raise TypeError(f"encode_bytes expects bytes/bytearray, got {type(data).__name__}")
        ids = list(data)
        if add_bos:
            ids.insert(0, self.BOS)
        if add_eos:
            ids.append(self.EOS)
        return ids

    def decode_bytes(self, ids: Any, strict: bool = False) -> bytes:
        """
        Inverse of encode_bytes. Strips special tokens, returns raw bytes
        with no UTF-8 interpretation — the only method that guarantees a
        lossless round trip for arbitrary binary input.
        """
        if hasattr(ids, "tolist"):
            ids = ids.tolist()
        elif not isinstance(ids, (list, tuple)):
            ids = list(ids)

        if strict:
            bad = [int(i) for i in ids if not (0 <= int(i) < self.VOCAB_SIZE)]
            if bad:
                raise ValueError(f"decode_bytes: invalid token id(s): {bad[:5]}"
                                  f"{'...' if len(bad) > 5 else ''}")
        return bytes(int(i) for i in ids if 0 <= int(i) <= 255)

    # ------------------------------------------------------------------ #
    # batch convenience
    # ------------------------------------------------------------------ #
    def encode_batch(self, texts: List[str], **kwargs) -> List[List[int]]:
        return [self.encode(t, **kwargs) for t in texts]

    def decode_batch(self, batches: List[List[int]], **kwargs) -> List[str]:
        return [self.decode(ids, **kwargs) for ids in batches]

    # ------------------------------------------------------------------ #
    # introspection
    # ------------------------------------------------------------------ #
    def is_special(self, token_id: int) -> bool:
        return token_id in self._SPECIAL_IDS

    def special_tokens(self) -> Tuple[int, int, int]:
        return (self.BOS, self.EOS, self.PAD)


# =============================================================================
# Configuration
# =============================================================================

@dataclass
class TensegrityConfig:
    vocab_size: int = 259
    block_size: int = 4096
    n_layer: int = 28
    n_embd: int = 2560
    n_head: int = 20
    n_kv_head: int = 4
    dropout: float = 0.0

    sliding_window: int = 1024
    global_every: int = 4
    rope_base: float = 12000.0

    ffn_mult: float = 8.0 / 3.0
    ffn_align: int = 64

    homeo_target: float = 1.0
    homeo_weight: float = 0.01
    gain_init: Optional[float] = None

    layerdrop_max: float = 0.1

    use_confidence_head: bool = True
    confidence_weight: float = 0.05

    bias: bool = False
    init_std: float = 0.02
    grad_checkpoint: bool = True
    grad_checkpoint_every: int = 1  # checkpoint every Nth block; 1 = current (all blocks) behavior

    # --- Optional upgrades, all off by default: an untouched config produces the
    # exact same model/forward behavior as before these fields existed. ---

    # Q/K RMSNorm before RoPE (LLaMA 3 / Gemma 2 / Cohere-style). Cures attention-logit
    # drift; adds 2 small RMSNorm modules (head_dim each) per attention block.
    use_qk_norm: bool = True

    # Z-loss on final logits (PaLM/Gemma 2): coef * mean(logsumexp(logits, dim=-1) ** 2),
    # added to total_loss. 0.0 disables the term entirely (not computed, not added).
    z_loss_weight: float = 0.0

    # Logit soft-capping (Gemma 2): logits = soft_cap * tanh(logits / soft_cap), applied
    # right before both the LM loss and generation. None disables it (no-op).
    logit_soft_cap: Optional[float] = None

    # Scale token embeddings by sqrt(n_embd) after lookup (Gemma 2 convention). Off by
    # default so old checkpoints' embedding scale/lm_head tying is unaffected.
    embed_scale: bool = True


    def __init__(self, *args, **kwargs):
        fields = dataclasses.fields(self)
        for f, val in zip(fields, args):
            kwargs[f.name] = val
        for f in fields:
            if f.name in kwargs:
                setattr(self, f.name, kwargs[f.name])
            elif f.default is not dataclasses.MISSING:
                setattr(self, f.name, f.default)
            elif f.default_factory is not dataclasses.MISSING:
                setattr(self, f.name, f.default_factory())
            else:
                raise TypeError(f"Missing required argument: {f.name}")
        self.__post_init__()

    def __post_init__(self):
        assert self.n_embd % self.n_head == 0, "n_embd must be divisible by n_head"
        head_dim = self.n_embd // self.n_head
        assert head_dim % 2 == 0, "head_dim (n_embd // n_head) must be even for RoPE"
        assert self.n_head % self.n_kv_head == 0, "n_head must be divisible by n_kv_head for GQA"
        assert self.global_every >= 1
        assert self.grad_checkpoint_every >= 1, "grad_checkpoint_every must be >= 1"
        assert 0.0 <= self.layerdrop_max < 1.0
        assert self.sliding_window >= 1
        if self.gain_init is None:
            self.gain_init = 1.0 / math.sqrt(2.0 * self.n_layer)


# =============================================================================
# Named configuration presets
# =============================================================================

# "base15m": ~15M total parameters (verified by instantiating TensegrityLM and
# summing param.numel() — see test_presets.py, which asserts this stays in
# [14.5e6, 16e6]). Intended as a small, fast-to-iterate reference scale for
# local experiments and notebooks. If you edit these kwargs, re-run
# test_presets.py to confirm the size didn't drift out of range.
BASE15M_CONFIG_KWARGS: Dict[str, Any] = dict(
    n_layer=4,
    n_embd=512,
    n_head=4,
    n_kv_head=2,
    ffn_mult=4.0,
    ffn_align=64,
    block_size=256,
    sliding_window=124,
    global_every=4,
)

BALANCED135M_CONFIG_KWARGS: Dict[str, Any] = dict(
    n_layer=12,
    n_embd=1024,
    n_head=16,
    n_kv_head=4,
    ffn_mult=8.0 / 3.0,
    ffn_align=64,
    block_size=2048,
    sliding_window=512,
    global_every=4,
)

HIGHINTEL400M_CONFIG_KWARGS: Dict[str, Any] = dict(
    n_layer=16,
    n_embd=1536,
    n_head=12,
    n_kv_head=4,
    ffn_mult=8.0 / 3.0,
    ffn_align=64,
    block_size=2048,
    sliding_window=512,
    global_every=4,
)

MODEL_PRESETS: Dict[str, Dict[str, Any]] = {
    "base15m": BASE15M_CONFIG_KWARGS,
    "balanced135m": BALANCED135M_CONFIG_KWARGS,
    "highintel400m": HIGHINTEL400M_CONFIG_KWARGS,
}


def get_preset_config(name: str, **overrides) -> "TensegrityConfig":
    """
    Build a TensegrityConfig from a named preset in MODEL_PRESETS, optionally
    overriding individual fields (e.g. get_preset_config("base15m", dropout=0.1)).
    """
    if name not in MODEL_PRESETS:
        raise ValueError(f"Unknown preset {name!r}; available presets: {sorted(MODEL_PRESETS)}")
    kwargs = dict(MODEL_PRESETS[name])
    kwargs.update(overrides)
    return TensegrityConfig(**kwargs)


# =============================================================================
# Core building blocks
# =============================================================================

class RMSNorm(nn.Module):
    def __init__(self, dim: int, eps: float = 1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        in_dtype = x.dtype
        x_fp32 = x.float()
        rms = torch.rsqrt(x_fp32.pow(2).mean(dim=-1, keepdim=True) + self.eps)
        out = x_fp32 * rms * self.weight.float()
        return out.to(in_dtype)


def precompute_rope(head_dim: int, max_seq_len: int, base: float = 10000.0,
                     device=None, dtype=torch.float32) -> Tuple[torch.Tensor, torch.Tensor]:
    """
    Precompute cos/sin tables for RoPE. Returns tensors of shape (max_seq_len, head_dim // 2).
    """
    assert head_dim % 2 == 0
    inv_freq = 1.0 / (base ** (torch.arange(0, head_dim, 2, device=device, dtype=torch.float32) / head_dim))
    t = torch.arange(max_seq_len, device=device, dtype=torch.float32)
    freqs = torch.outer(t, inv_freq)  # (max_seq_len, head_dim // 2)
    cos = freqs.cos().to(dtype)
    sin = freqs.sin().to(dtype)
    return cos, sin


def apply_rope(x: torch.Tensor, cos: torch.Tensor, sin: torch.Tensor) -> torch.Tensor:
    B, H, T, D = x.shape
    half = D // 2
    assert T <= cos.shape[0], (
        f"sequence length {T} exceeds precomputed RoPE table length {cos.shape[0]}; "
        f"call precompute_rope with a larger max_seq_len"
    )
    in_dtype = x.dtype
    x_fp32 = x.float()
    x1 = x_fp32[..., :half]
    x2 = x_fp32[..., half:]
    cos_ = cos[:T].float().view(1, 1, T, half)
    sin_ = sin[:T].float().view(1, 1, T, half)
    out1 = x1 * cos_ - x2 * sin_
    out2 = x2 * cos_ + x1 * sin_
    return torch.cat([out1, out2], dim=-1).to(in_dtype)


def activation_rms(x: torch.Tensor, eps: float = 1e-8) -> torch.Tensor:
    """Mean RMS of activations across the feature dimension, averaged over batch/tokens.
    Stabilized with eps to avoid sqrt(0) singularity and NaN gradients."""
    return torch.sqrt(x.float().pow(2).mean(dim=-1) + eps).mean()


def apply_logit_soft_cap(logits: torch.Tensor, soft_cap: float) -> torch.Tensor:
    """Gemma-2-style logit soft-capping: soft_cap * tanh(logits / soft_cap). Bounds
    logits to (-soft_cap, soft_cap) with a smooth (not hard-clipped) falloff."""
    in_dtype = logits.dtype
    capped = soft_cap * torch.tanh(logits.float() / soft_cap)
    return capped.to(in_dtype)


# =============================================================================
# Attention (Grouped-Query, with alternating local/global masks)
# =============================================================================

class CausalSelfAttention(nn.Module):
    def __init__(self, cfg: TensegrityConfig, is_global: bool):
        super().__init__()
        assert cfg.n_embd % cfg.n_head == 0, (
            f"n_embd ({cfg.n_embd}) must be divisible by n_head ({cfg.n_head}); "
            f"otherwise head_dim truncates silently and o_proj/q_proj shapes desync."
        )
        assert cfg.n_head % cfg.n_kv_head == 0, (
            f"n_head ({cfg.n_head}) must be divisible by n_kv_head ({cfg.n_kv_head}) for GQA grouping."
        )
        self.n_head = cfg.n_head
        self.n_kv_head = cfg.n_kv_head
        self.head_dim = cfg.n_embd // cfg.n_head
        self.n_rep = cfg.n_head // cfg.n_kv_head
        self.is_global = is_global
        self.sliding_window = cfg.sliding_window
        self.dropout_p = cfg.dropout
        self.q_proj = nn.Linear(cfg.n_embd, cfg.n_head * self.head_dim, bias=cfg.bias)
        self.k_proj = nn.Linear(cfg.n_embd, cfg.n_kv_head * self.head_dim, bias=cfg.bias)
        self.v_proj = nn.Linear(cfg.n_embd, cfg.n_kv_head * self.head_dim, bias=cfg.bias)
        self.o_proj = nn.Linear(cfg.n_head * self.head_dim, cfg.n_embd, bias=cfg.bias)
        self.resid_dropout = nn.Dropout(cfg.dropout)
        self._chunk_plan = None
        self._chunk_plan_T = None
        self._chunk_plan_device = None

        # Q/K RMSNorm (LLaMA 3 / Gemma 2 / Cohere): per-head normalization applied
        # right after projection, before RoPE. Off by default -- modules aren't even
        # created, so a checkpoint saved with use_qk_norm=False has no q_norm/k_norm
        # keys, matching a model built with the same (default) config exactly.
        if cfg.use_qk_norm:
            self.q_norm = RMSNorm(self.head_dim)
            self.k_norm = RMSNorm(self.head_dim)
        else:
            self.q_norm = None
            self.k_norm = None

    def forward(self, x: torch.Tensor, cos: torch.Tensor, sin: torch.Tensor) -> torch.Tensor:
        # No mask parameter here: global layers get an exact causal mask via
        # is_causal=True, and local layers get one via _local_attention's chunking --
        # neither ever materializes a (T, T) score matrix. (forward_incremental, the
        # separate KV-cached path used by generate(), does take an explicit mask --
        # it needs one for prefill, where there's no is_causal-style shortcut once a
        # cache is involved.)
        B, T, C = x.shape
        q = self.q_proj(x).view(B, T, self.n_head, self.head_dim).transpose(1, 2)       # (B, nH, T, D)
        k = self.k_proj(x).view(B, T, self.n_kv_head, self.head_dim).transpose(1, 2)    # (B, nKV, T, D)
        v = self.v_proj(x).view(B, T, self.n_kv_head, self.head_dim).transpose(1, 2)    # (B, nKV, T, D)
        if self.q_norm is not None:
            q = self.q_norm(q)
            k = self.k_norm(k)
        q = apply_rope(q, cos, sin)
        k = apply_rope(k, cos, sin)

        dropout_p = self.dropout_p if self.training else 0.0
        if self.n_rep > 1 and _SDPA_SUPPORTS_GQA:
            gqa_kwargs = {"enable_gqa": True}
        else:
            gqa_kwargs = {}
            if self.n_rep > 1:
                # Fallback for torch < 2.5.
                k = k.repeat_interleave(self.n_rep, dim=1)
                v = v.repeat_interleave(self.n_rep, dim=1)

        if self.is_global:
            # No custom mask needed: is_causal=True is exactly the "global" mask
            # and lets SDPA pick its fused Flash/memory-efficient kernel instead
            # of materializing a (B, nH, T, T) score matrix.
            y = F.scaled_dot_product_attention(q, k, v, dropout_p=dropout_p, is_causal=True, **gqa_kwargs)
        else:
            y = self._local_attention(q, k, v, dropout_p, gqa_kwargs)

        y = y.transpose(1, 2).contiguous().view(B, T, self.n_head * self.head_dim)
        y = self.o_proj(y)
        y = self.resid_dropout(y)
        return y

    def _local_attention(self, q: torch.Tensor, k: torch.Tensor, v: torch.Tensor,
                          dropout_p: float, gqa_kwargs: dict) -> torch.Tensor:
        """
        Sliding-window causal attention, computed one window-sized chunk of
        queries at a time against only the keys/values that chunk could
        legally attend to. Equivalent to masking a full (T, T) causal matrix
        with `j > i - sliding_window` (the old behavior), but never
        materializes that (T, T) tensor -- each chunk's score matrix is at
        most (sliding_window, 2 * sliding_window - 1).
        """
        W = self.sliding_window
        T = q.shape[2]
        plan = self._get_local_chunk_plan(T, W, q.device)
        outs = []
        for q_start, q_end, k_start, mask in plan:
            qc = q[:, :, q_start:q_end, :]
            kc = k[:, :, k_start:q_end, :]
            vc = v[:, :, k_start:q_end, :]
            if mask is None:
                outs.append(F.scaled_dot_product_attention(
                    qc, kc, vc, dropout_p=dropout_p, is_causal=True, **gqa_kwargs
                ))
            else:
                outs.append(F.scaled_dot_product_attention(
                    qc, kc, vc, attn_mask=mask, dropout_p=dropout_p, **gqa_kwargs
                ))
        return torch.cat(outs, dim=2)

    def _get_local_chunk_plan(self, T: int, W: int, device: Any):
        """
        Cached list of (q_start, q_end, k_start, mask) chunks covering [0, T)
        in windows of size W. k_start == q_start for the first chunk (its
        window covers the whole chunk, so mask=None + is_causal=True is
        exact); later chunks need trailing keys from the previous chunk too,
        so they get an explicit small mask. Cache is keyed on (T, device) and
        is safe to reuse across grad-checkpoint recomputation since it's a
        pure function of those two values.
        """
        device = torch.device(device)
        if (self._chunk_plan is not None and self._chunk_plan_T == T
                and self._chunk_plan_device == device):
            return self._chunk_plan

        plan = []
        n_chunks = (T + W - 1) // W
        for c in range(n_chunks):
            q_start, q_end = c * W, min((c + 1) * W, T)
            k_start = max(0, q_start - W + 1)
            if k_start == q_start:
                plan.append((q_start, q_end, k_start, None))
                continue
            qlen = q_end - q_start
            offset = q_start - k_start
            i = torch.arange(qlen, device=device).view(-1, 1)
            j = torch.arange(offset + qlen, device=device).view(1, -1)
            mask = (j <= i + offset) & (j > i - W + offset)
            plan.append((q_start, q_end, k_start, mask))

        self._chunk_plan = plan
        self._chunk_plan_T = T
        self._chunk_plan_device = device
        return plan

    def forward_incremental(
        self, x: torch.Tensor, cos: torch.Tensor, sin: torch.Tensor,
        past_kv: Optional[Tuple[torch.Tensor, torch.Tensor]] = None,
        attn_mask: Optional[torch.Tensor] = None,
    ) -> Tuple[torch.Tensor, Tuple[torch.Tensor, torch.Tensor]]:
        """
        KV-cached counterpart to forward(), used only by TensegrityLM.generate().
        Does not touch or share code with forward() -- training always goes
        through forward() unchanged.

        Two supported call patterns:
          - Prefill: attn_mask given (the usual boolean local/global mask,
            sliced to the chunk being processed), past_kv=None. x holds a
            whole new chunk (e.g. the prompt). Attention-wise this is
            identical to forward(); it additionally returns the (k, v) cache
            needed to seed subsequent decode steps.
          - Decode: attn_mask=None, past_kv from a previous call, x is a
            single new token (T == 1). No mask is applied because every key
            retained in the cache is causally at or before the query position
            by construction, so there is nothing to mask out.

        For local (sliding-window) layers, k/v -- both what's attended to and
        what's returned as the new cache -- are trimmed to the trailing
        `sliding_window` entries, since no future query can ever reach further
        back than that regardless of how long generation continues.
        """
        B, T, C = x.shape
        q = self.q_proj(x).view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_kv_head, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_kv_head, self.head_dim).transpose(1, 2)
        if self.q_norm is not None:
            q = self.q_norm(q)
            k = self.k_norm(k)
        q = apply_rope(q, cos, sin)
        k = apply_rope(k, cos, sin)

        if attn_mask is not None:
            # Prefill: attn_mask already encodes the local/global windowing across
            # all T query positions, so k/v must stay at their full T length here
            # -- trimming before this call would desync them from the mask's key
            # dimension (each row of the mask expects to see the *whole* chunk).
            assert past_kv is None, "prefill (attn_mask given) must not also receive a past_kv cache"
        else:
            assert T == 1, "no-mask incremental attention only supports single-token decode steps"
            if past_kv is not None:
                past_k, past_v = past_kv
                k = torch.cat([past_k, k], dim=2)
                v = torch.cat([past_v, v], dim=2)
            # No mask enforces the window here, so it must be applied directly:
            # trim to the trailing `sliding_window` keys/values before attending.
            if not self.is_global and k.shape[2] > self.sliding_window:
                k = k[:, :, -self.sliding_window:, :].contiguous()
                v = v[:, :, -self.sliding_window:, :].contiguous()

        if self.n_rep > 1:
            if _SDPA_SUPPORTS_GQA:
                y = F.scaled_dot_product_attention(
                    q, k, v, attn_mask=attn_mask, dropout_p=0.0, is_causal=False, enable_gqa=True,
                )
            else:
                y = F.scaled_dot_product_attention(
                    q, k.repeat_interleave(self.n_rep, dim=1), v.repeat_interleave(self.n_rep, dim=1),
                    attn_mask=attn_mask, dropout_p=0.0, is_causal=False,
                )
        else:
            y = F.scaled_dot_product_attention(q, k, v, attn_mask=attn_mask, dropout_p=0.0, is_causal=False)

        y = y.transpose(1, 2).contiguous().view(B, T, self.n_head * self.head_dim)
        y = self.o_proj(y)

        # Cache handed back to the caller: local layers only ever need the
        # trailing `sliding_window` keys/values regardless of how long this
        # chunk was (a long prefill chunk gets trimmed down here; a decode
        # step was already trimmed above, so this is a no-op for it).
        if not self.is_global and k.shape[2] > self.sliding_window:
            new_cache = (k[:, :, -self.sliding_window:, :].contiguous(), v[:, :, -self.sliding_window:, :].contiguous())
        else:
            new_cache = (k.contiguous(), v.contiguous())

        return y, new_cache


def build_attention_masks(block_size: int, sliding_window: int, device=None) -> Dict[str, torch.Tensor]:
    """
    Build boolean attention masks of shape (block_size, block_size).
    True = positions that may attend to each other (query i attends to key j).
    """
    idx = torch.arange(block_size, device=device)
    i = idx.view(-1, 1)
    j = idx.view(1, -1)
    causal = j <= i
    local = causal & (j > (i - sliding_window))
    return {"global": causal, "local": local}


# =============================================================================
# Feed-forward: SwiGLU
# =============================================================================

def ffn_hidden_dim(n_embd: int, mult: float, align: int) -> int:
    assert n_embd > 0, f"n_embd must be positive, got {n_embd}"
    assert mult > 0, f"mult must be positive, got {mult}"
    assert align >= 1, f"align must be a positive integer, got {align}"
    hidden = int(n_embd * mult)
    hidden = align * ((hidden + align - 1) // align)
    return max(hidden, align)


class SwiGLU(nn.Module):
    def __init__(self, n_embd: int, hidden_dim: int, bias: bool, dropout: float):
        super().__init__()
        self.w_gate = nn.Linear(n_embd, hidden_dim, bias=bias)
        self.w_up = nn.Linear(n_embd, hidden_dim, bias=bias)
        self.w_down = nn.Linear(hidden_dim, n_embd, bias=bias)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.dropout(self.w_down(F.silu(self.w_gate(x)) * self.w_up(x)))




# =============================================================================
# Transformer block
# =============================================================================

class Block(nn.Module):
    def __init__(self, cfg: TensegrityConfig, layer_idx: int):
        super().__init__()
        assert cfg.global_every > 0, f"global_every must be positive, got {cfg.global_every}"
        assert cfg.n_layer >= 1, f"n_layer must be >= 1, got {cfg.n_layer}"
        assert 0.0 <= cfg.layerdrop_max <= 1.0, (
            f"layerdrop_max must be in [0, 1], got {cfg.layerdrop_max}"
        )

        self.layer_idx = layer_idx
        is_global = ((layer_idx + 1) % cfg.global_every == 0)

        self.ln1 = RMSNorm(cfg.n_embd)
        self.attn = CausalSelfAttention(cfg, is_global=is_global)
        self.ln2 = RMSNorm(cfg.n_embd)

        hidden_dim = ffn_hidden_dim(cfg.n_embd, cfg.ffn_mult, cfg.ffn_align)
        self.ffn = SwiGLU(cfg.n_embd, hidden_dim, cfg.bias, cfg.dropout)

        # Homeostatic residual gains (learnable per-layer, per-sublayer scalars)
        self.attn_gain = nn.Parameter(torch.tensor(float(cfg.gain_init)))
        self.ffn_gain = nn.Parameter(torch.tensor(float(cfg.gain_init)))
        self.homeo_target = cfg.homeo_target

        # Depth-scaled stochastic depth probability
        n_layer = cfg.n_layer
        if n_layer > 1:
            self.drop_p = cfg.layerdrop_max * (layer_idx / (n_layer - 1))
        else:
            self.drop_p = 0.0

    def forward(self, x: torch.Tensor, cos: torch.Tensor, sin: torch.Tensor,
                ) -> Tuple[torch.Tensor, torch.Tensor, bool]:
        # Stochastic depth: during training, randomly skip the entire block (batch-level)
        if self.training and self.drop_p > 0.0:
            skip = torch.rand((), device=x.device)
            # Synchronize the drop decision across DDP ranks. Without this, ranks
            # can independently choose different skip/no-skip paths, which desyncs
            # the set of parameters that receive gradients this step and can hang
            # or silently corrupt DDP's gradient all-reduce. No-op outside DDP.
            if dist.is_available() and dist.is_initialized():
                dist.broadcast(skip, src=0)
            # This .item() sync was already happening here regardless (it's how the
            # skip/no-skip decision itself gets made), so surfacing it as the
            # return value below adds no new synchronization point.
            if skip.item() < self.drop_p:
                zero = x.new_zeros((), dtype=torch.float32)  # match dtype of the non-dropped path below
                return x, zero, True

        attn_out = self.attn(self.ln1(x), cos, sin)
        homeo_attn = (activation_rms(attn_out) - self.homeo_target).pow(2)
        x = x + self.attn_gain * attn_out

        ffn_out = self.ffn(self.ln2(x))
        homeo_ffn = (activation_rms(ffn_out) - self.homeo_target).pow(2)
        x = x + self.ffn_gain * ffn_out

        homeo = homeo_attn + homeo_ffn
        return x, homeo, False

    def forward_incremental(
        self, x: torch.Tensor, cos: torch.Tensor, sin: torch.Tensor,
        past_kv: Optional[Tuple[torch.Tensor, torch.Tensor]],
        attn_mask: Optional[torch.Tensor] = None,
    ) -> Tuple[torch.Tensor, Tuple[torch.Tensor, torch.Tensor]]:
        """
        KV-cached counterpart to forward(), used only by TensegrityLM.generate().
        No LayerDrop (generate() always runs in eval(), where drop_p never
        triggers anyway -- see the `self.training and ...` guard above) and no
        homeostatic-loss bookkeeping (inference reports no losses).
        """
        attn_out, new_kv = self.attn.forward_incremental(self.ln1(x), cos, sin, past_kv, attn_mask)
        x = x + self.attn_gain * attn_out

        ffn_out = self.ffn(self.ln2(x))
        x = x + self.ffn_gain * ffn_out

        return x, new_kv


# =============================================================================
# Muon optimizer (single-device variant)
# =============================================================================
#
# Vendored from https://github.com/KellerJordan/Muon (muon.py, SingleDeviceMuonWithAuxAdam),
# the reference implementation behind the modded-nanogpt speedrun results, adapted to drop
# the torch.distributed all_gather logic that variant needs for multi-GPU training (this repo
# targets a single GPU). zeropower_via_newtonschulz5's coefficients and muon_update's momentum/
# scaling formulas are reproduced verbatim -- these are exactly the numbers the speedrun results
# are evidence for, not values to casually retune. Off by default (TensegrityLM.configure_optimizer
# use_muon=False): produces the identical AdamW-only optimizer as before this was added.
#
# Muon should only be used for hidden weight layers (attention/FFN projection matrices).
# The token embedding (tied to lm_head), the confidence head, and every 1D parameter (RMSNorm
# weights, the homeostatic attn_gain/ffn_gain scalars) still go through AdamW -- see
# TensegrityLM.configure_optimizer's partitioning logic below for how that split is drawn for
# this specific model.

def zeropower_via_newtonschulz5(G: torch.Tensor, steps: int = 5) -> torch.Tensor:
    """
    Newton-Schulz iteration computing an approximate orthogonalization of G (the nearest
    matrix with all singular values ~1, not an exact SVD-based projection -- see upstream
    docstring: this produces US'V^T with S'_ii ~ Uniform(0.5, 1.5), which is empirically
    just as good for this use case as exact orthogonalization and much cheaper).
    """
    assert G.ndim >= 2
    a, b, c = (3.4445, -4.7750, 2.0315)
    X = G.bfloat16() if (G.is_cuda and torch.cuda.is_bf16_supported()) else G.float()
    if G.size(-2) > G.size(-1):
        X = X.mT
    X = X / (X.norm(dim=(-2, -1), keepdim=True) + 1e-7)
    for _ in range(steps):
        A = X @ X.mT
        B = b * A + c * A @ A
        X = a * X + B @ X
    if G.size(-2) > G.size(-1):
        X = X.mT
    return X.to(G.dtype)


def muon_update(grad: torch.Tensor, momentum: torch.Tensor, beta: float = 0.95,
                 ns_steps: int = 5, nesterov: bool = True) -> torch.Tensor:
    momentum.lerp_(grad, 1 - beta)
    update = grad.lerp_(momentum, beta) if nesterov else momentum
    if update.ndim == 4:  # conv filters, not used by this model but kept for fidelity to upstream
        update = update.view(len(update), -1)
    update = zeropower_via_newtonschulz5(update, steps=ns_steps)
    update = update * max(1.0, update.size(-2) / update.size(-1)) ** 0.5
    return update


def adam_update(grad: torch.Tensor, buf1: torch.Tensor, buf2: torch.Tensor, step: int,
                 betas: Tuple[float, float], eps: float) -> torch.Tensor:
    buf1.lerp_(grad, 1 - betas[0])
    buf2.lerp_(grad.square(), 1 - betas[1])
    buf1c = buf1 / (1 - betas[0] ** step)
    buf2c = buf2 / (1 - betas[1] ** step)
    return buf1c / (buf2c.sqrt() + eps)


class SingleDeviceMuonWithAuxAdam(torch.optim.Optimizer):
    """
    Single-GPU combined optimizer: runs Muon on param groups with use_muon=True, AdamW
    (reimplemented inline, matching upstream) on groups with use_muon=False. One optimizer
    object so the training loop only has one .step()/.zero_grad() to call, same as a plain
    AdamW would need.
    """
    def __init__(self, param_groups: List[Dict[str, Any]]):
        for group in param_groups:
            assert "use_muon" in group
            if group["use_muon"]:
                group["lr"] = group.get("lr", 0.02)
                group["momentum"] = group.get("momentum", 0.95)
                group["weight_decay"] = group.get("weight_decay", 0)
                assert set(group.keys()) == {"params", "lr", "momentum", "weight_decay", "use_muon"}
            else:
                group["lr"] = group.get("lr", 3e-4)
                group["betas"] = group.get("betas", (0.9, 0.95))
                group["eps"] = group.get("eps", 1e-10)
                group["weight_decay"] = group.get("weight_decay", 0)
                assert set(group.keys()) == {"params", "lr", "betas", "eps", "weight_decay", "use_muon"}
        super().__init__(param_groups, dict())

    @torch.no_grad()
    def step(self, closure=None):
        loss = None
        if closure is not None:
            with torch.enable_grad():
                loss = closure()

        for group in self.param_groups:
            if group["use_muon"]:
                for p in group["params"]:
                    if p.grad is None:
                        continue
                    state = self.state[p]
                    if len(state) == 0:
                        state["momentum_buffer"] = torch.zeros_like(p)
                    update = muon_update(p.grad, state["momentum_buffer"], beta=group["momentum"])
                    p.mul_(1 - group["lr"] * group["weight_decay"])
                    p.add_(update.reshape(p.shape), alpha=-group["lr"])
            else:
                for p in group["params"]:
                    if p.grad is None:
                        continue
                    state = self.state[p]
                    if len(state) == 0:
                        state["exp_avg"] = torch.zeros_like(p)
                        state["exp_avg_sq"] = torch.zeros_like(p)
                        state["step"] = 0
                    state["step"] += 1
                    update = adam_update(p.grad, state["exp_avg"], state["exp_avg_sq"],
                                          state["step"], group["betas"], group["eps"])
                    p.mul_(1 - group["lr"] * group["weight_decay"])
                    p.add_(update, alpha=-group["lr"])

        return loss


# =============================================================================
# Full model
# =============================================================================

class TensegrityLM(nn.Module):
    def __init__(self, cfg: TensegrityConfig):
        super().__init__()
        assert cfg.n_layer >= 1, f"n_layer must be >= 1, got {cfg.n_layer}"

        self.cfg = cfg
        self.grad_checkpoint = cfg.grad_checkpoint
        self.grad_checkpoint_every = cfg.grad_checkpoint_every

        self.token_emb = nn.Embedding(cfg.vocab_size, cfg.n_embd)
        self.drop = nn.Dropout(cfg.dropout)
        self.blocks = nn.ModuleList([Block(cfg, i) for i in range(cfg.n_layer)])
        self.ln_f = RMSNorm(cfg.n_embd)
        self.lm_head = nn.Linear(cfg.n_embd, cfg.vocab_size, bias=False)

        # Weight tying
        self.token_emb.weight = self.lm_head.weight

        # Optional confidence head: predicts P(argmax prediction == target)
        if cfg.use_confidence_head:
            self.confidence_head = nn.Linear(cfg.n_embd, 1, bias=True)
        else:
            self.confidence_head = None

        head_dim = cfg.n_embd // cfg.n_head
        cos, sin = precompute_rope(head_dim, cfg.block_size, base=cfg.rope_base)
        self.register_buffer("rope_cos", cos, persistent=False)
        self.register_buffer("rope_sin", sin, persistent=False)

        # Attention masks are built lazily (see _get_masks) sized to the actual T of each
        # forward call rather than eagerly materialized at the full (block_size, block_size)
        # here -- for large block_size that dense boolean pair is O(block_size^2) memory
        # that's wasteful since T is almost always block_size at training time anyway (so the
        # lazily-built mask ends up the same size) and can be much smaller at eval/generation
        # time. A single-entry cache (keyed on T) avoids rebuilding every step in the common
        # case where T doesn't change between calls.
        self._mask_cache_T: Optional[int] = None
        self._mask_cache: Optional[Dict[str, torch.Tensor]] = None

        self.apply(self._init_weights)

        n_params = sum(p.numel() for p in self.parameters())
        print(f"[TensegrityLM] layers={cfg.n_layer} "
              f"n_embd={cfg.n_embd} n_head={cfg.n_head} n_kv_head={cfg.n_kv_head} "
              f"vocab={cfg.vocab_size} block_size={cfg.block_size}")
        print(f"[TensegrityLM] total parameters: {n_params:,} ({n_params/1e6:.2f}M)")

    def _init_weights(self, module: nn.Module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=self.cfg.init_std)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=self.cfg.init_std)

    def _get_masks(self, T: int, device: torch.device) -> Dict[str, torch.Tensor]:
        """
        Return the (T, T) global/local boolean masks, rebuilding only when T (or device)
        differs from the last call. build_attention_masks(T, ...) is exactly equal to
        build_attention_masks(block_size, ...)[:T, :T] (the causal/window relation only
        depends on the query/key indices themselves, not on block_size), so this is a
        drop-in replacement for the old eagerly-materialized block_size-sized buffers.
        """
        cached = self._mask_cache
        if self._mask_cache_T != T or cached is None or cached["global"].device != device:
            cached = build_attention_masks(T, self.cfg.sliding_window, device=device)
            self._mask_cache = cached
            self._mask_cache_T = T
        return cached

    def forward(self, idx: torch.Tensor, targets: Optional[torch.Tensor] = None
                 ) -> Tuple[torch.Tensor, Optional[Dict[str, torch.Tensor]], Optional[torch.Tensor]]:
        """
        idx:     (B, T) integer token ids, T <= block_size
        targets: (B, T) integer token ids (next-token targets), or None for inference

        Returns:
            logits: (B, T, vocab_size)
            losses: dict of scalar tensors (only if targets is not None), else None
            confidence: (B, T) sigmoid confidence scores (only if targets is None and head enabled), else None
        """
        B, T = idx.shape
        assert T <= self.cfg.block_size, f"sequence length {T} exceeds block_size {self.cfg.block_size}"

        x = self.token_emb(idx)  # (B, T, C)
        if self.cfg.embed_scale:
            x = x * (self.cfg.n_embd ** 0.5)
        x = self.drop(x)

        cos = self.rope_cos[:T]
        sin = self.rope_sin[:T]

        # Explicit fp32: Block returns fp32 homeo, so keep the running total fp32 from
        # the start rather than relying on implicit type promotion on the first add.
        total_homeo = x.new_zeros((), dtype=torch.float32)
        n_active = 0

        use_checkpoint = self.grad_checkpoint and self.training and x.requires_grad

        for i, block in enumerate(self.blocks):
            # grad_checkpoint_every == 1 (default) checkpoints every block, identical to prior
            # behavior. Larger values checkpoint only every Nth block (by index), trading some
            # of the memory savings back for fewer recomputation passes during backward.
            if use_checkpoint and i % self.grad_checkpoint_every == 0:
                x, homeo, was_skipped = torch.utils.checkpoint.checkpoint(
                    block, x, cos, sin, use_reentrant=False
                )
            else:
                x, homeo, was_skipped = block(x, cos, sin)
            total_homeo = total_homeo + homeo
            if not was_skipped:
                n_active += 1

        x = self.ln_f(x)
        logits = self.lm_head(x)
        if self.cfg.logit_soft_cap is not None:
            logits = apply_logit_soft_cap(logits, self.cfg.logit_soft_cap)

        if targets is not None:
            ce_loss = F.cross_entropy(
                logits.view(-1, logits.size(-1)), targets.view(-1),
                ignore_index=-1,
            )

            denom = max(1, n_active * 2)
            avg_homeo = total_homeo / denom

            losses = {
                "ce_loss": ce_loss,
                "homeo_loss": avg_homeo,
            }

            total_loss = ce_loss + self.cfg.homeo_weight * avg_homeo

            if self.cfg.z_loss_weight > 0.0:
                # PaLM / Gemma 2 z-loss: penalizes logsumexp(logits) drifting away from 0,
                # which keeps the softmax normalizer well-scaled and prevents logit blowup.
                z_loss = self.cfg.z_loss_weight * torch.logsumexp(logits.float(), dim=-1).pow(2).mean()
                losses["z_loss"] = z_loss
                total_loss = total_loss + z_loss

            if self.confidence_head is not None:
                conf_logits = self.confidence_head(x).squeeze(-1)  # (B, T)
                with torch.no_grad():
                    preds = logits.argmax(dim=-1)  # (B, T)
                    correct = (preds == targets).float()
                    valid = (targets != -1).float()
                conf_loss_raw = F.binary_cross_entropy_with_logits(
                    conf_logits, correct, reduction="none"
                )
                denom_c = valid.sum().clamp_min(1.0)
                conf_loss = (conf_loss_raw * valid).sum() / denom_c
                losses["confidence_loss"] = conf_loss
                total_loss = total_loss + self.cfg.confidence_weight * conf_loss

            losses["total_loss"] = total_loss
            return logits, losses, None

        else:
            confidence = None
            if self.confidence_head is not None:
                conf_logits = self.confidence_head(x).squeeze(-1)  # (B, T)
                confidence = torch.sigmoid(conf_logits)
            return logits, None, confidence

    def forward_incremental(
        self, idx: torch.Tensor,
        past_kv_list: Optional[List[Optional[Tuple[torch.Tensor, torch.Tensor]]]] = None,
        pos_start: int = 0,
    ) -> Tuple[torch.Tensor, List[Tuple[torch.Tensor, torch.Tensor]], Optional[torch.Tensor]]:
        """
        KV-cached counterpart to forward(), used only by generate() for
        autoregressive sampling. Entirely separate code path from forward():
        grad_checkpoint, homeostatic loss, and LayerDrop are a
        training-time concern that forward() still handles exactly as before;
        none of that applies here since generate() always runs the model in
        eval() under @torch.no_grad().

        idx: (B, T) ids for the NEW tokens being fed this call -- the whole
             prompt on the first (prefill) call, a single token (T == 1) on
             every subsequent (decode) call.
        past_kv_list: None on the prefill call; otherwise the list this method
             returned on the previous call (one (k, v) cache per block).
        pos_start: absolute position of idx[:, 0] in the full generated
             sequence. Needed because, unlike forward() (which always starts
             at position 0), idx here is typically a suffix of the sequence,
             so the RoPE table must be sliced at an offset.

        Returns (logits, new_past_kv_list, confidence): logits/confidence
        cover only the T new positions, mirroring forward()'s shapes.
        """
        B, T = idx.shape
        is_prefill = past_kv_list is None
        if is_prefill:
            past_kv_list = [None] * len(self.blocks)

        x = self.token_emb(idx)
        if self.cfg.embed_scale:
            x = x * (self.cfg.n_embd ** 0.5)
        x = self.drop(x)

        cos = self.rope_cos[pos_start:pos_start + T]
        sin = self.rope_sin[pos_start:pos_start + T]

        if is_prefill:
            masks = self._get_masks(T, x.device)
            mask_global = masks["global"]
            mask_local = masks["local"]
        else:
            mask_global = None
            mask_local = None

        new_past_kv_list = []
        for block, past_kv in zip(self.blocks, past_kv_list):
            attn_mask = mask_global if block.attn.is_global else mask_local
            x, new_kv = block.forward_incremental(x, cos, sin, past_kv, attn_mask)
            new_past_kv_list.append(new_kv)

        x = self.ln_f(x)
        logits = self.lm_head(x)
        if self.cfg.logit_soft_cap is not None:
            logits = apply_logit_soft_cap(logits, self.cfg.logit_soft_cap)

        confidence = None
        if self.confidence_head is not None:
            conf_logits = self.confidence_head(x).squeeze(-1)  # (B, T)
            confidence = torch.sigmoid(conf_logits)

        return logits, new_past_kv_list, confidence

    # -------------------------------------------------------------------
    # Generation
    # -------------------------------------------------------------------
    @torch.no_grad()
    def generate(self, idx: torch.Tensor, max_new_tokens: int,
                  temperature: float = 1.0, top_k: Optional[int] = None,
                  top_p: Optional[float] = None,
                  use_confidence_temp: bool = False,
                  conf_temp_min: float = 0.5, conf_temp_max: float = 1.5,
                  stop_on_eos: bool = True, use_cache: bool = True) -> torch.Tensor:
        """
        idx: (B, T) starting context.
        Returns (B, T + max_new_tokens) — may be shorter if EOS is hit and stop_on_eos
        is True, in which case generation stops once every sequence in the batch has
        produced EOS (finished sequences are padded with EOS in the interim, not
        re-sampled).

        use_cache: if True (default), reuse a per-layer KV cache
            (forward_incremental) across decode steps instead of recomputing
            attention over the whole context on every new token. This is a
            pure performance optimization -- it produces bit-identical token
            sequences to use_cache=False for the same seed/inputs (see
            test_kv_cache.py) -- not an approximation, so it's safe to leave
            on by default. If the running sequence ever exceeds cfg.block_size,
            the cache is transparently rebuilt from the truncated trailing
            window, matching the sliding-window behavior of the non-cached path.
        """
        assert temperature >= 0.0, f"temperature must be >= 0, got {temperature}"
        assert top_k is None or top_k > 0, f"top_k must be None or positive, got {top_k}"

        # generate() must not permanently change the model's mode: if this is called
        # mid-training (e.g. periodic sampling), leaving the model in eval() afterward
        # silently disables dropout/stochastic depth for the rest of training with no
        # error anywhere.
        was_training = self.training
        self.eval()
        try:
            B = idx.size(0)
            finished = torch.zeros(B, dtype=torch.bool, device=idx.device) if stop_on_eos else None
            past_kv_list = None

            for _ in range(max_new_tokens):
                T = idx.size(1)
                if use_cache:
                    if past_kv_list is None or T > self.cfg.block_size:
                        # Prefill: either the very first step, or the running sequence
                        # just outgrew block_size and the cache must be rebuilt from the
                        # truncated trailing window (same sliding-window fallback the
                        # non-cached path below uses on every single step).
                        idx_cond = idx if T <= self.cfg.block_size else idx[:, -self.cfg.block_size:]
                        logits, past_kv_list, confidence = self.forward_incremental(idx_cond, None, 0)
                    else:
                        # Steady-state decode: only the just-sampled token is new; its
                        # absolute position is T - 1 (idx already includes it).
                        logits, past_kv_list, confidence = self.forward_incremental(idx[:, -1:], past_kv_list, T - 1)
                    logits = logits[:, -1, :]  # (B, V)
                else:
                    idx_cond = idx if T <= self.cfg.block_size else idx[:, -self.cfg.block_size:]
                    logits, _, confidence = self.forward(idx_cond, targets=None)
                    logits = logits[:, -1, :]  # (B, V)

                greedy = False
                if temperature == 0.0:
                    # Greedy decoding. An explicit temperature=0 request must win over
                    # confidence-temp scaling below — otherwise a caller with
                    # use_confidence_temp=True and an active confidence head could never
                    # get deterministic output, even when explicitly asking for it.
                    # (Also: temperature=0 previously divided by zero -> NaN logits ->
                    # multinomial errors out; this is a common expected convention instead.)
                    greedy = True
                elif use_confidence_temp and confidence is not None:
                    conf = confidence[:, -1]  # (B,)
                    # Per-sequence temperature scaling: low confidence -> higher temperature, scaled by base temperature
                    scale = conf_temp_max - conf * (conf_temp_max - conf_temp_min)  # (B,)
                    eff_temp = (temperature * scale).clamp_min(1e-4)
                    logits = logits / eff_temp.unsqueeze(-1)
                else:
                    logits = logits / max(temperature, 1e-4)

                if greedy:
                    next_tok = logits.argmax(dim=-1, keepdim=True)
                else:
                    if top_k is not None:
                        v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                        logits[logits < v[:, [-1]]] = -float("inf")

                    if top_p is not None:
                        sorted_logits, sorted_idx = torch.sort(logits, descending=True, dim=-1)
                        probs = F.softmax(sorted_logits, dim=-1)
                        cumprobs = torch.cumsum(probs, dim=-1)
                        cutoff = cumprobs > top_p
                        cutoff[:, 1:] = cutoff[:, :-1].clone()
                        cutoff[:, 0] = False
                        sorted_logits = sorted_logits.masked_fill(cutoff, -float("inf"))
                        logits = torch.full_like(logits, -float("inf")).scatter(1, sorted_idx, sorted_logits)

                    probs = F.softmax(logits, dim=-1)
                    # Guard against degenerate sampling distributions (underflow or all-inf masking)
                    invalid_probs = torch.isnan(probs).any(dim=-1) | (probs.sum(dim=-1) <= 0.0)
                    if invalid_probs.any():
                        greedy_tok = logits.argmax(dim=-1, keepdim=True)
                        probs_safe = torch.nan_to_num(probs, nan=0.0)
                        sum_p = probs_safe.sum(dim=-1, keepdim=True)
                        zero_rows = sum_p.squeeze(-1) <= 0.0
                        probs_safe = torch.where(zero_rows.unsqueeze(-1), torch.ones_like(probs_safe) / probs_safe.size(-1), probs_safe / sum_p.clamp_min(1e-12))
                        sampled_tok = torch.multinomial(probs_safe, num_samples=1)
                        next_tok = torch.where(invalid_probs.unsqueeze(-1), greedy_tok, sampled_tok)
                    else:
                        next_tok = torch.multinomial(probs, num_samples=1)  # (B, 1)

                if stop_on_eos:
                    # Freeze already-finished sequences instead of continuing to sample
                    # them, so a fast-finishing sequence doesn't keep drifting while
                    # waiting on slower ones in the same batch.
                    next_tok = torch.where(
                        finished.unsqueeze(-1),
                        torch.full_like(next_tok, ByteTokenizer.EOS),
                        next_tok,
                    )

                idx = torch.cat([idx, next_tok], dim=1)

                if stop_on_eos:
                    finished = finished | (next_tok.squeeze(-1) == ByteTokenizer.EOS)
                    if finished.all():
                        break

            return idx
        finally:
            self.train(was_training)

    # -------------------------------------------------------------------
    # Optimizer configuration
    # -------------------------------------------------------------------
    def configure_optimizer(self, weight_decay: float, learning_rate: float,
                             betas: Tuple[float, float], device_type: str,
                             use_8bit: bool = False,
                             use_muon: bool = False,
                             muon_lr: float = 0.02,
                             muon_momentum: float = 0.95) -> torch.optim.Optimizer:
        """
        use_muon: off by default (identical AdamW-only optimizer as before this flag
            existed). When True, returns a SingleDeviceMuonWithAuxAdam instead: every
            ndim>=2 parameter belonging to self.blocks (attention q/k/v/o_proj, SwiGLU
            weights, and Q/K norm
            weights are ndim==1 so they don't qualify) goes to Muon; everything else --
            the tied token_emb/lm_head weight, the confidence head, and every 1D parameter
            (RMSNorm weights, the homeostatic attn_gain/ffn_gain scalars) -- goes to AdamW
            in the same optimizer object. This split follows Muon's own usage guidance
            (see the class docstring above): it's meant for hidden weight layers only, not
            embeddings/output heads/scalars.
        """
        if use_muon:
            return self._configure_muon_optimizer(weight_decay, learning_rate, betas,
                                                    muon_lr, muon_momentum)

        decay_params = []
        no_decay_params = []
        seen = set()
        for name, p in self.named_parameters():
            if not p.requires_grad:
                continue
            if id(p) in seen:
                continue
            seen.add(id(p))
            if p.dim() >= 2:
                decay_params.append(p)
            else:
                no_decay_params.append(p)

        groups = [
            {"params": decay_params, "weight_decay": weight_decay},
            {"params": no_decay_params, "weight_decay": 0.0},
        ]

        # Low-memory 8-bit AdamW path: same defensive try/import-or-fall-back pattern as
        # the fused-AdamW detection below. AdamW keeps 2 fp32 moment tensors per param
        # (exp_avg, exp_avg_sq) -- 8 bytes/param of pure optimizer state on top of the
        # model weights themselves; bitsandbytes' AdamW8bit quantizes those moments to
        # int8 (~2 bytes/param), a ~4x reduction in optimizer-state memory. Only
        # attempted on CUDA (bitsandbytes' 8-bit kernels are CUDA-only); any ImportError
        # or non-CUDA device silently falls back to the existing torch.optim.AdamW path.
        if use_8bit and device_type == "cuda" and torch.cuda.is_available():
            try:
                import bitsandbytes as bnb
                optimizer = bnb.optim.AdamW8bit(groups, lr=learning_rate, betas=betas)
                return optimizer
            except ImportError:
                pass

        # Robust fused-AdamW feature detection: __code__.co_varnames breaks on any
        # PyTorch build where __init__ is wrapped by a decorator that doesn't preserve
        # __code__ faithfully. inspect.signature + try/except mirrors the SDPA
        # enable_gqa feature-detection used in CausalSelfAttention. Also confirm CUDA
        # is actually available rather than trusting the device_type string alone —
        # otherwise fused=True can be handed to an optimizer with no CUDA tensors and
        # fail confusingly at the first .step() instead of here.
        use_fused = False
        if device_type == "cuda" and torch.cuda.is_available():
            try:
                use_fused = "fused" in inspect.signature(torch.optim.AdamW).parameters
            except (TypeError, ValueError):
                use_fused = False

        extra = {"fused": True} if use_fused else {}
        optimizer = torch.optim.AdamW(groups, lr=learning_rate, betas=betas, **extra)
        return optimizer

    def _configure_muon_optimizer(self, weight_decay: float, learning_rate: float,
                                   betas: Tuple[float, float],
                                   muon_lr: float, muon_momentum: float) -> torch.optim.Optimizer:
        seen = set()
        muon_params = []
        for name, p in self.blocks.named_parameters():
            if not p.requires_grad or id(p) in seen:
                continue
            if p.dim() >= 2:
                muon_params.append(p)
                seen.add(id(p))

        # Everything not already claimed by the Muon pass above: the tied token_emb/lm_head
        # weight, the confidence head, and every 1D param (norms, homeostatic gains) --
        # deliberately walking self.named_parameters() (the whole model) a second time
        # rather than special-casing by name, so nothing new added to the model in the
        # future silently falls through uncovered by either group.
        adamw_decay_params = []
        adamw_no_decay_params = []
        for name, p in self.named_parameters():
            if not p.requires_grad or id(p) in seen:
                continue
            seen.add(id(p))
            if p.dim() >= 2:
                adamw_decay_params.append(p)
            else:
                adamw_no_decay_params.append(p)

        param_groups = [
            dict(params=muon_params, lr=muon_lr, momentum=muon_momentum,
                 weight_decay=weight_decay, use_muon=True),
            dict(params=adamw_decay_params, lr=learning_rate, betas=betas,
                 eps=1e-10, weight_decay=weight_decay, use_muon=False),
            dict(params=adamw_no_decay_params, lr=learning_rate, betas=betas,
                 eps=1e-10, weight_decay=0.0, use_muon=False),
        ]
        return SingleDeviceMuonWithAuxAdam(param_groups)


# =============================================================================
# Data pipeline
# =============================================================================

def read_file_to_bytes(filepath: str) -> bytes:
    """
    Reads a file and converts its textual contents to bytes.
    Supports .txt, .md, .parquet, and .jsonl formats.
    """
    ext = os.path.splitext(filepath)[1].lower()
    if ext == ".parquet":
        try:
            import pandas as pd
            df = pd.read_parquet(filepath)
            str_cols = [c for c in df.columns if pd.api.types.is_string_dtype(df[c]) or df[c].dtype == object]
            lines = []
            for col in str_cols:
                lines.extend(df[col].dropna().astype(str).tolist())
            content = "\n\n".join(lines)
            return content.encode("utf-8", errors="replace")
        except Exception as e:
            print(f"[warning] Failed to parse parquet {filepath}: {e}, falling back to raw read")
            with open(filepath, "rb") as fh:
                return fh.read()
    elif ext == ".jsonl":
        lines = []
        try:
            with open(filepath, "r", encoding="utf-8", errors="replace") as fh:
                for line in fh:
                    line = line.strip()
                    if not line:
                        continue
                    try:
                        data = json.loads(line)
                        if isinstance(data, dict):
                            val = data.get("text") or data.get("content") or data.get("body")
                            if val is None:
                                val = "\n".join(str(v) for v in data.values() if isinstance(v, (str, int, float)))
                            if val:
                                lines.append(str(val))
                        elif isinstance(data, str):
                            lines.append(data)
                    except Exception:
                        pass
            content = "\n\n".join(lines)
            return content.encode("utf-8", errors="replace")
        except Exception as e:
            print(f"[warning] Failed to parse jsonl {filepath}: {e}, falling back to raw read")
            with open(filepath, "rb") as fh:
                return fh.read()
    else:
        with open(filepath, "rb") as fh:
            return fh.read()


def prepare_data(data_path: str, out_dir: str, val_fraction: float = 0.1, seed: int = 1337,
                  shuffle_files: bool = False):
    """
    Read text, markdown, parquet, or jsonl file(s) recursively, byte-encode (with EOS
    separators between documents), split into train/val, and write uint16 .bin files.
    """
    os.makedirs(out_dir, exist_ok=True)
    tok = ByteTokenizer()

    supported_exts = {".txt", ".md", ".parquet", ".jsonl"}

    if os.path.isdir(data_path):
        all_files = sorted(glob.glob(os.path.join(data_path, "**", "*"), recursive=True))
        files = [f for f in all_files if os.path.isfile(f) and (os.path.splitext(f)[1].lower() in supported_exts or not supported_exts)]
        if not files:  # fallback to all files if no specific extension matches
            files = [f for f in all_files if os.path.isfile(f)]
    else:
        files = [data_path]

    if not files:
        raise ValueError(f"No files found at {data_path}")

    if shuffle_files:
        files = list(files)
        random.Random(seed).shuffle(files)

    # Streaming data preparation: determine file sizes and write file-by-file
    file_lengths = []
    for f in files:
        raw_bytes = read_file_to_bytes(f)
        file_lengths.append(len(raw_bytes))
        del raw_bytes

    n = sum(file_lengths) + len(files)  # + 1 EOS token per file
    if n == 0:
        raise ValueError(f"All files at {data_path} are empty; nothing to tokenize.")

    n_val = max(1, int(n * val_fraction))
    n_train = n - n_val
    if n_train <= 0:
        raise ValueError("Dataset too small after train/val split; provide more data.")

    train_path = os.path.join(out_dir, "train.bin")
    val_path = os.path.join(out_dir, "val.bin")

    train_mm = np.memmap(train_path, dtype=np.uint16, mode="w+", shape=(n_train,))
    val_mm = np.memmap(val_path, dtype=np.uint16, mode="w+", shape=(n_val,))

    pos = 0  # position in the full concatenated token stream

    def write_chunk(chunk_ids: np.ndarray):
        nonlocal pos
        chunk_len = len(chunk_ids)
        chunk_start = pos
        chunk_end = pos + chunk_len
        train_end = min(chunk_end, n_train)
        if train_end > chunk_start:
            train_mm[chunk_start:train_end] = chunk_ids[: train_end - chunk_start]
        if chunk_end > n_train:
            val_start = max(chunk_start, n_train)
            val_mm[val_start - n_train: chunk_end - n_train] = chunk_ids[val_start - chunk_start:]
        pos = chunk_end

    eos_chunk = np.array([ByteTokenizer.EOS], dtype=np.uint16)
    for f in files:
        raw = read_file_to_bytes(f)
        # Vectorized zero-copy uint8 -> uint16 token conversion (70x faster than list-based encoding)
        ids = np.frombuffer(raw, dtype=np.uint8).astype(np.uint16)
        write_chunk(ids)
        write_chunk(eos_chunk)
        del raw, ids

    train_mm.flush()
    val_mm.flush()
    del train_mm, val_mm

    meta = {
        "vocab_size": ByteTokenizer.VOCAB_SIZE,
        "n_train_tokens": int(n_train),
        "n_val_tokens": int(n_val),
        "source_files": files,
        "val_fraction": val_fraction,
        "shuffle_files": shuffle_files,
        "seed": seed if shuffle_files else None,
    }
    with open(os.path.join(out_dir, "meta.json"), "w") as f:
        json.dump(meta, f, indent=2)

    print(f"[prepare] processed {len(files)} files recursively across .txt/.md/.parquet/.jsonl")
    print(f"[prepare] wrote {n_train:,} train tokens -> {train_path}")
    print(f"[prepare] wrote {n_val:,} val tokens -> {val_path}")


_MEMMAP_CACHE: Dict[str, np.memmap] = {}


def _get_memmap(path: str) -> np.memmap:
    """Cached np.memmap open. Data files are read-only and don't change during
    training, so reusing the same memmap across calls is safe and avoids a fresh
    open()+mmap() syscall on every single get_batch/get_batch_mixed call."""
    arr = _MEMMAP_CACHE.get(path)
    if arr is None:
        arr = np.memmap(path, dtype=np.uint16, mode="r")
        _MEMMAP_CACHE[path] = arr
    return arr


def get_batch(data_dir: str, split: str, batch_size: int, block_size: int,
               device: str) -> Tuple[torch.Tensor, torch.Tensor]:
    path = os.path.join(data_dir, f"{split}.bin")
    arr = _get_memmap(path)
    n = len(arr)
    if n <= block_size + 1:
        raise ValueError(
            f"{split}.bin has only {n} tokens, need > block_size+1 ({block_size + 1}). "
            f"Use a larger corpus or smaller --block_size."
        )
    max_start = n - block_size - 1
    starts = np.random.randint(0, max_start, size=(batch_size,))
    # Single-gather memmap consolidation: fetch (block_size + 1) contiguous tokens per row
    # in one gather instead of two separate strided memmap passes, cutting disk reads in half.
    offsets = np.arange(block_size + 1)
    idx_xy = starts[:, None] + offsets[None, :]  # (batch_size, block_size + 1)
    xy = arr[idx_xy].astype(np.int64)
    x = np.ascontiguousarray(xy[:, :-1])
    y = np.ascontiguousarray(xy[:, 1:])
    x_t = torch.from_numpy(x)
    y_t = torch.from_numpy(y)
    # startswith, not ==: DDP/multi-GPU runs commonly pass "cuda:0", "cuda:1", etc.,
    # which previously fell through to the no-pin-memory branch silently.
    if device.startswith("cuda"):
        x_t = x_t.pin_memory().to(device, non_blocking=True)
        y_t = y_t.pin_memory().to(device, non_blocking=True)
    else:
        x_t = x_t.to(device)
        y_t = y_t.to(device)
    return x_t, y_t


# =============================================================================
# Checkpointing
# =============================================================================

CHECKPOINT_FORMAT_VERSION = 1  # bump when the checkpoint schema changes; load_checkpoint warns on mismatch


def save_checkpoint(path: str, model: nn.Module, optimizer: torch.optim.Optimizer,
                     cfg: TensegrityConfig, step: int, best_val_loss: float,
                     rng_state: Dict[str, Any], scaler: Optional[torch.amp.GradScaler] = None):
    """
    scaler: pass the training GradScaler (if fp16 mixed precision is in use) so its
        loss-scale/growth-tracker state survives resume. Default None preserves prior
        checkpoint schema exactly for callers that don't pass it. If scaler is passed
        but disabled (e.g. bf16/fp32 training), its state_dict is still saved (it's a
        cheap no-op dict) for schema consistency across checkpoints from the same run.
    """
    raw_model = model.module if hasattr(model, "module") else model

    out_dir = os.path.dirname(path)
    if out_dir:
        os.makedirs(out_dir, exist_ok=True)

    ckpt = {
        "format_version": CHECKPOINT_FORMAT_VERSION,
        "model_state_dict": raw_model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scaler_state_dict": scaler.state_dict() if scaler is not None else None,
        "config": asdict(cfg),
        "step": step,
        "best_val_loss": best_val_loss,
        "rng_state": rng_state,
    }
    tmp_path = path + ".tmp"
    torch.save(ckpt, tmp_path)
    # Flush to disk before the atomic rename, so a crash/power-loss immediately after
    # os.replace can't leave a checkpoint that *looks* complete (the rename succeeded)
    # but whose contents were still sitting in OS write-back cache.
    with open(tmp_path, "r+b") as f:
        os.fsync(f.fileno())
    os.replace(tmp_path, path)


def load_checkpoint(path: str, device: str) -> Dict[str, Any]:
    ckpt = torch.load(path, map_location=device, weights_only=False)
    required = ("model_state_dict", "optimizer_state_dict", "config", "step", "best_val_loss")
    missing = [k for k in required if k not in ckpt]
    if missing:
        raise ValueError(
            f"{path} is missing expected checkpoint key(s) {missing}; "
            f"this may not be a TensegrityLM checkpoint, or it predates the current format."
        )
    ckpt_version = ckpt.get("format_version")
    if ckpt_version != CHECKPOINT_FORMAT_VERSION:
        print(f"[warn] {path} has format_version={ckpt_version!r}, code expects "
              f"{CHECKPOINT_FORMAT_VERSION!r}. Loading will proceed, but the schema may "
              f"have changed since this checkpoint was saved -- if load_state_dict below "
              f"fails or behaves oddly, this mismatch is why.")
    return ckpt


def capture_rng_state() -> Dict[str, Any]:
    state = {
        "torch": torch.get_rng_state(),
        "numpy": np.random.get_state(),
        "random": random.getstate(),
    }
    if torch.cuda.is_available():
        state["cuda"] = torch.cuda.get_rng_state_all()
    return state


def restore_rng_state(state: Dict[str, Any]):
    if state is None:
        return
    if "random" in state:
        try:
            random.setstate(state["random"])
        except Exception as e:
            print(f"[warn] failed to restore python random state ({e})")
    if "torch" in state:
        torch_state = state["torch"]
        if isinstance(torch_state, torch.Tensor):
            torch_state = torch_state.cpu()
        torch.set_rng_state(torch_state)
    if "numpy" in state:
        np.random.set_state(state["numpy"])
    if "cuda" in state and torch.cuda.is_available():
        try:
            cuda_state = state["cuda"]
            if isinstance(cuda_state, (list, tuple)):
                cuda_state = [s.cpu() if isinstance(s, torch.Tensor) else s for s in cuda_state]
            n_avail = torch.cuda.device_count()
            if len(cuda_state) == n_avail:
                torch.cuda.set_rng_state_all(cuda_state)
            else:
                # Safely restore available GPUs individually if device counts differ across machines
                for dev_id in range(min(len(cuda_state), n_avail)):
                    torch.cuda.set_rng_state(cuda_state[dev_id], device=dev_id)
        except Exception as e:
            print(f"[warn] failed to restore CUDA RNG state ({e})")


# =============================================================================
# Learning rate schedule (cosine with warmup — "temperature annealing")
# =============================================================================

def get_lr(step: int, warmup_steps: int, max_steps: int, max_lr: float, min_lr: float) -> float:
    if step < warmup_steps:
        return max_lr * (step + 1) / warmup_steps
    if step >= max_steps:
        return min_lr
    decay_ratio = (step - warmup_steps) / max(1, (max_steps - warmup_steps))
    coeff = 0.5 * (1.0 + math.cos(math.pi * decay_ratio))
    return min_lr + coeff * (max_lr - min_lr)


def get_lr_wsd(step: int, warmup_steps: int, stable_steps: int, decay_steps: int,
               max_lr: float, min_lr: float) -> float:
    """
    Warmup-Stable-Decay (MiniCPM; also used by Llama 3.1, DeepSeek-V2): linear warmup to
    max_lr, hold flat at max_lr for stable_steps, then linearly decay to min_lr over
    decay_steps. Unlike get_lr's cosine schedule, decay only has to be as long as
    decay_steps -- you don't have to commit to a total token budget upfront, since "when do
    I start decaying" is a decision you can make later by just choosing when this function's
    stable phase ends, not something baked into the schedule shape from step 0.

    step is measured from 0; the three phases are
        [0, warmup_steps), [warmup_steps, warmup_steps + stable_steps),
        [warmup_steps + stable_steps, warmup_steps + stable_steps + decay_steps),
    and any step beyond that returns min_lr, same convention as get_lr.
    """
    assert warmup_steps >= 1, "warmup_steps must be >= 1"
    assert decay_steps >= 1, "decay_steps must be >= 1"
    decay_start = warmup_steps + stable_steps
    decay_end = decay_start + decay_steps

    if step < warmup_steps:
        return max_lr * (step + 1) / warmup_steps
    if step < decay_start:
        return max_lr
    if step >= decay_end:
        return min_lr
    decay_ratio = (step - decay_start) / decay_steps
    return max_lr - decay_ratio * (max_lr - min_lr)


# =============================================================================
# Device / dtype helpers
# =============================================================================

def pick_device(requested: str = "auto") -> str:
    if requested != "auto":
        return requested
    if torch.cuda.is_available():
        return "cuda"
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return "mps"
    return "cpu"


def pick_dtype(requested: str, device: str) -> torch.dtype:
    if requested != "auto":
        return {"float32": torch.float32, "bfloat16": torch.bfloat16, "float16": torch.float16}[requested]
    if device == "cuda" and torch.cuda.is_bf16_supported():
        return torch.bfloat16
    if device == "cuda":
        return torch.float16
    return torch.float32


# =============================================================================
# Training
# =============================================================================

def build_config_from_args(args) -> TensegrityConfig:
    cfg_kwargs = {}
    preset_name = getattr(args, "preset", None)
    if preset_name is not None:
        cfg_kwargs.update(MODEL_PRESETS[preset_name])
    for f in dataclasses.fields(TensegrityConfig):
        if hasattr(args, f.name):
            val = getattr(args, f.name)
            if val is not None:
                cfg_kwargs[f.name] = val
    return TensegrityConfig(**cfg_kwargs)


def cmd_train(args):
    is_ddp = int(os.environ.get("RANK", -1)) != -1
    if is_ddp:
        from torch.nn.parallel import DistributedDataParallel as DDP
        dist.init_process_group(backend="nccl")
        ddp_rank = int(os.environ["RANK"])
        ddp_local_rank = int(os.environ["LOCAL_RANK"])
        ddp_world_size = int(os.environ["WORLD_SIZE"])
        device = f"cuda:{ddp_local_rank}"
        torch.cuda.set_device(device)
        master_process = (ddp_rank == 0)
        seed_offset = ddp_rank
    else:
        ddp_world_size = 1
        master_process = True
        seed_offset = 0
        device = pick_device(args.device)

    torch.manual_seed(args.seed + seed_offset)
    np.random.seed(args.seed + seed_offset)

    os.makedirs(args.out_dir, exist_ok=True)

    meta_path = os.path.join(args.data_dir, "meta.json")
    if os.path.exists(meta_path):
        with open(meta_path) as f:
            meta = json.load(f)
        if master_process:
            print(f"[data] {meta['n_train_tokens']:,} train tokens, "
                  f"{meta['n_val_tokens']:,} val tokens, vocab={meta['vocab_size']}")

    device_type = "cuda" if "cuda" in device else ("mps" if device == "mps" else "cpu")
    dtype = pick_dtype(args.dtype, device_type)
    if master_process:
        print(f"[train] device={device} dtype={dtype}")

    ckpt_resume = None
    if args.init_from == "resume":
        latest_path = os.path.join(args.out_dir, "ckpt_latest.pt")
        if os.path.exists(latest_path):
            ckpt_resume = load_checkpoint(latest_path, device)
            if master_process:
                print(f"[resume] loaded checkpoint from {latest_path} at step {ckpt_resume['step']}")
        else:
            if master_process:
                print(f"[resume] no checkpoint found at {latest_path}, starting from scratch")

    if ckpt_resume is not None:
        cfg = TensegrityConfig(**ckpt_resume["config"])
    else:
        cfg = build_config_from_args(args)

    model = TensegrityLM(cfg)
    model.to(device)

    if ckpt_resume is not None:
        model.load_state_dict(ckpt_resume["model_state_dict"])

    optimizer = model.configure_optimizer(
        weight_decay=args.weight_decay, learning_rate=args.learning_rate,
        betas=(args.beta1, args.beta2), device_type=device_type,
        use_8bit=args.use_8bit_optimizer,
        use_muon=getattr(args, "use_muon", False),
        muon_lr=getattr(args, "muon_lr", 0.02),
        muon_momentum=getattr(args, "muon_momentum", 0.95),
    )

    # BUGFIX: scaler must exist before the resume block below can call
    # scaler.load_state_dict(...) on it. It previously was constructed ~30 lines further
    # down, after the resume block already referenced it — an UnboundLocalError waiting
    # to happen the moment scaler_state_dict was actually populated in a checkpoint (see
    # the save_checkpoint calls further down, which previously never passed scaler=scaler
    # at all, so this path was silently never exercised).
    use_amp = dtype in (torch.float16, torch.bfloat16) and device_type in ("cuda", "cpu")
    amp_dtype = dtype if use_amp else torch.float32
    scaler = torch.amp.GradScaler(enabled=(dtype == torch.float16 and device_type == "cuda"))

    if ckpt_resume is not None:
        optimizer.load_state_dict(ckpt_resume["optimizer_state_dict"])
        restore_rng_state(ckpt_resume.get("rng_state"))
        if ckpt_resume.get("scaler_state_dict") is not None:
            scaler.load_state_dict(ckpt_resume["scaler_state_dict"])

    # Checkpoints are saved with the step number that was just completed (see the
    # save_checkpoint calls below, which happen before `step += 1`), so resuming
    # must continue from the next step, not repeat the one already trained.
    start_step = (ckpt_resume["step"] + 1) if ckpt_resume is not None else 0
    best_val_loss = ckpt_resume["best_val_loss"] if ckpt_resume is not None else float("inf")

    raw_model = model
    if args.compile:
        try:
            model = torch.compile(model)
            if master_process:
                print("[train] torch.compile enabled")
        except Exception as e:
            if master_process:
                print(f"[train] torch.compile failed ({e}), continuing without it")

    if is_ddp:
        # NOTE: find_unused_parameters=True is required because LayerDrop may skip
        # entire layers, leaving their params without gradients. This adds overhead.
        model = DDP(model, device_ids=[ddp_local_rank], find_unused_parameters=True)
        raw_model = model.module

    log_path = os.path.join(args.out_dir, "train_log.jsonl")
    log_file = open(log_path, "a") if master_process else None

    try:
        t0 = time.time()
        step = start_step
        running_loss = 0.0
        steps_since_log = 0

        model.train()
        optimizer.zero_grad(set_to_none=True)

        while step < args.max_steps:
            lr = get_lr(step, args.warmup_steps, args.max_steps, args.learning_rate, args.min_lr)
            for group in optimizer.param_groups:
                group["lr"] = lr

            for micro_step in range(args.grad_accum_steps):
                x, y = get_batch(args.data_dir, "train", args.batch_size, cfg.block_size, device)

                if is_ddp:
                    model.require_backward_grad_sync = (micro_step == args.grad_accum_steps - 1)

                with torch.amp.autocast(device_type=device_type if device_type != "mps" else "cpu",
                                          dtype=amp_dtype, enabled=use_amp):
                    logits, losses, _ = model(x, y)
                    loss = losses["total_loss"] / args.grad_accum_steps

                if scaler.is_enabled():
                    scaler.scale(loss).backward()
                else:
                    loss.backward()

                running_loss += losses["total_loss"].item() / args.grad_accum_steps

            steps_since_log += 1

            if args.grad_clip > 0.0:
                if scaler.is_enabled():
                    scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), args.grad_clip)

            if scaler.is_enabled():
                scaler.step(optimizer)
                scaler.update()
            else:
                optimizer.step()
            optimizer.zero_grad(set_to_none=True)

            if step % args.log_interval == 0 and master_process:
                dt = time.time() - t0
                avg_loss = running_loss / max(1, steps_since_log)
                ce = losses["ce_loss"].item()
                homeo = losses["homeo_loss"].item()
                conf = losses.get("confidence_loss")
                conf_str = f" conf={conf.item():.4f}" if conf is not None else ""
                print(f"step {step:6d} | lr {lr:.2e} | total {losses['total_loss'].item():.4f} "
                      f"(ce {ce:.4f} homeo {homeo:.4f}{conf_str}) | {dt:.1f}s")
                log_file.write(json.dumps({
                    "step": step, "lr": lr, "total_loss": losses["total_loss"].item(),
                    "ce_loss": ce, "homeo_loss": homeo,
                    "time": dt,
                }) + "\n")
                log_file.flush()
                running_loss = 0.0
                steps_since_log = 0

            if step > 0 and step % args.eval_interval == 0 and master_process:
                val_loss = evaluate(raw_model, args.data_dir, cfg, device, device_type, amp_dtype,
                                      use_amp, args.eval_iters)
                print(f"step {step:6d} | val_loss {val_loss:.4f}")
                log_file.write(json.dumps({"step": step, "val_loss": val_loss}) + "\n")
                log_file.flush()

                rng_state = capture_rng_state()
                save_checkpoint(os.path.join(args.out_dir, "ckpt_latest.pt"), raw_model, optimizer,
                                 cfg, step, best_val_loss, rng_state, scaler=scaler)
                if val_loss < best_val_loss:
                    best_val_loss = val_loss
                    save_checkpoint(os.path.join(args.out_dir, "ckpt_best.pt"), raw_model, optimizer,
                                     cfg, step, best_val_loss, rng_state, scaler=scaler)
                    print(f"step {step:6d} | new best val_loss {best_val_loss:.4f}, checkpoint saved")

            step += 1

        if master_process:
            val_loss = evaluate(raw_model, args.data_dir, cfg, device, device_type, amp_dtype,
                                  use_amp, args.eval_iters)
            rng_state = capture_rng_state()
            save_checkpoint(os.path.join(args.out_dir, "ckpt_latest.pt"), raw_model, optimizer,
                             cfg, step, best_val_loss, rng_state, scaler=scaler)
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                save_checkpoint(os.path.join(args.out_dir, "ckpt_best.pt"), raw_model, optimizer,
                                 cfg, step, best_val_loss, rng_state, scaler=scaler)
            print(f"[train] done. final val_loss={val_loss:.4f} best_val_loss={best_val_loss:.4f}")
    finally:
        if log_file is not None:
            log_file.close()

    if is_ddp:
        dist.destroy_process_group()


@torch.no_grad()
def evaluate(model: nn.Module, data_dir: str, cfg: TensegrityConfig, device: str,
             device_type: str, amp_dtype: torch.dtype, use_amp: bool, eval_iters: int) -> float:
    was_training = model.training
    model.eval()
    try:
        losses = []
        for split in ["val"]:
            for _ in range(eval_iters):
                x, y = get_batch(data_dir, split, batch_size=8, block_size=cfg.block_size, device=device)
                with torch.amp.autocast(device_type=device_type if device_type != "mps" else "cpu",
                                          dtype=amp_dtype, enabled=use_amp):
                    _, loss_dict, _ = model(x, y)
                losses.append(loss_dict["ce_loss"].item())
        return float(np.mean(losses))
    finally:
        if was_training:
            model.train()


# =============================================================================
# Generation / Eval CLI commands
# =============================================================================

def cmd_generate(args):
    device = pick_device(args.device)
    ckpt = load_checkpoint(args.ckpt, device)
    cfg = TensegrityConfig(**ckpt["config"])
    model = TensegrityLM(cfg)
    model.load_state_dict(ckpt["model_state_dict"])
    model.to(device)
    model.eval()

    tok = ByteTokenizer()
    if args.seed is not None:
        torch.manual_seed(args.seed)

    if args.prompt:
        ids = tok.encode(args.prompt)
    else:
        ids = []
    if args.add_bos:
        ids = [ByteTokenizer.BOS] + ids
    if not ids:
        ids = [ByteTokenizer.BOS]

    idx = torch.tensor([ids], dtype=torch.long, device=device)

    out = model.generate(
        idx, max_new_tokens=args.max_new_tokens, temperature=args.temperature,
        top_k=args.top_k, top_p=args.top_p,
        use_confidence_temp=args.use_confidence_temp,
        stop_on_eos=not args.no_stop_on_eos,
        use_cache=not args.no_kv_cache,
    )

    out_ids = out[0].tolist()
    text = tok.decode(out_ids)
    print(text)


def cmd_eval(args):
    device = pick_device(args.device)
    ckpt = load_checkpoint(args.ckpt, device)
    cfg = TensegrityConfig(**ckpt["config"])
    model = TensegrityLM(cfg)
    model.load_state_dict(ckpt["model_state_dict"])
    model.to(device)
    model.eval()

    device_type = "cuda" if "cuda" in device else ("mps" if device == "mps" else "cpu")
    dtype = pick_dtype(args.dtype, device_type)
    use_amp = dtype in (torch.float16, torch.bfloat16) and device_type in ("cuda", "cpu")

    val_loss = evaluate(model, args.data_dir, cfg, device, device_type, dtype, use_amp, args.eval_iters)
    print(f"val_loss = {val_loss:.4f}")
    print(f"val_perplexity = {math.exp(val_loss):.4f}")


# =============================================================================
# Argument parsing
# =============================================================================

def add_model_args(p: argparse.ArgumentParser):
    p.add_argument("--preset", type=str, default=None, choices=sorted(MODEL_PRESETS),
                   help="Apply a named model-size preset (e.g. base15m, ~15M params) "
                        "as config defaults; any other --<field> flag passed explicitly "
                        "still overrides the preset's value for that field.")
    p.add_argument("--block_size", type=int, default=None)
    p.add_argument("--n_layer", type=int, default=None)
    p.add_argument("--n_embd", type=int, default=None)
    p.add_argument("--n_head", type=int, default=None)
    p.add_argument("--n_kv_head", type=int, default=None)
    p.add_argument("--dropout", type=float, default=None)
    p.add_argument("--sliding_window", type=int, default=None)
    p.add_argument("--global_every", type=int, default=None)
    p.add_argument("--rope_base", type=float, default=None)
    p.add_argument("--ffn_mult", type=float, default=None)
    p.add_argument("--ffn_align", type=int, default=None)
    p.add_argument("--homeo_target", type=float, default=None)
    p.add_argument("--homeo_weight", type=float, default=None)
    p.add_argument("--layerdrop_max", type=float, default=None)
    p.add_argument("--use_confidence_head", type=lambda x: x.lower() == "true", default=None)
    p.add_argument("--confidence_weight", type=float, default=None)
    p.add_argument("--bias", type=lambda x: x.lower() == "true", default=None)
    p.add_argument("--grad_checkpoint", type=lambda x: x.lower() == "true", default=None)
    p.add_argument("--grad_checkpoint_every", type=int, default=None)
    p.add_argument("--use_qk_norm", type=lambda x: x.lower() == "true", default=None,
                   help="Per-head RMSNorm on queries and keys before RoPE.")
    p.add_argument("--z_loss_weight", type=float, default=None,
                   help="Auxiliary z-loss weight on final logits.")
    p.add_argument("--logit_soft_cap", type=float, default=None,
                   help="Gemma-2-style logit soft capping value (e.g. 30.0).")
    p.add_argument("--embed_scale", type=lambda x: x.lower() == "true", default=None,
                   help="Scale token embeddings by sqrt(n_embd).")


def main():
    parser = argparse.ArgumentParser(description="TensegrityLM — byte-level LM, single-file implementation")
    sub = parser.add_subparsers(dest="command", required=True)

    # prepare
    p_prep = sub.add_parser("prepare", help="Tokenize a text file/directory into train/val .bin files")
    p_prep.add_argument("--data_path", type=str, required=True)
    p_prep.add_argument("--out_dir", type=str, required=True)
    p_prep.add_argument("--val_fraction", type=float, default=0.1)
    p_prep.add_argument("--seed", type=int, default=1337)

    # train
    p_train = sub.add_parser("train", help="Train (or resume) a model")
    p_train.add_argument("--data_dir", type=str, required=True)
    p_train.add_argument("--out_dir", type=str, required=True)
    p_train.add_argument("--init_from", type=str, default="scratch", choices=["scratch", "resume"])
    p_train.add_argument("--device", type=str, default="auto")
    p_train.add_argument("--dtype", type=str, default="auto", choices=["auto", "float32", "bfloat16", "float16"])
    p_train.add_argument("--seed", type=int, default=1337)
    p_train.add_argument("--batch_size", type=int, default=12)
    p_train.add_argument("--grad_accum_steps", type=int, default=1)
    p_train.add_argument("--max_steps", type=int, default=5000)
    p_train.add_argument("--warmup_steps", type=int, default=100)
    p_train.add_argument("--learning_rate", type=float, default=3e-4)
    p_train.add_argument("--min_lr", type=float, default=3e-5)
    p_train.add_argument("--weight_decay", type=float, default=0.1)
    p_train.add_argument("--beta1", type=float, default=0.9)
    p_train.add_argument("--beta2", type=float, default=0.95)
    p_train.add_argument("--use_8bit_optimizer", action="store_true",
                         help="Use bitsandbytes' 8-bit AdamW to cut optimizer-state memory "
                              "(~4x smaller moments); silently falls back to regular AdamW "
                              "if bitsandbytes isn't installed or device is not CUDA.")
    p_train.add_argument("--use_muon", action="store_true",
                         help="Use Muon optimizer for hidden matrix weights (attention/FFN).")
    p_train.add_argument("--muon_lr", type=float, default=0.02,
                         help="Learning rate for Muon parameter groups (default: 0.02).")
    p_train.add_argument("--muon_momentum", type=float, default=0.95,
                         help="Momentum coefficient for Muon optimizer (default: 0.95).")
    p_train.add_argument("--grad_clip", type=float, default=1.0)
    p_train.add_argument("--log_interval", type=int, default=10)
    p_train.add_argument("--eval_interval", type=int, default=250)
    p_train.add_argument("--eval_iters", type=int, default=20)
    p_train.add_argument("--compile", action="store_true")
    add_model_args(p_train)

    # generate
    p_gen = sub.add_parser("generate", help="Generate text from a checkpoint")
    p_gen.add_argument("--ckpt", type=str, required=True)
    p_gen.add_argument("--device", type=str, default="auto")
    p_gen.add_argument("--prompt", type=str, default="")
    p_gen.add_argument("--add_bos", action="store_true")
    p_gen.add_argument("--max_new_tokens", type=int, default=200)
    p_gen.add_argument("--temperature", type=float, default=0.8)
    p_gen.add_argument("--top_k", type=int, default=None)
    p_gen.add_argument("--top_p", type=float, default=None)
    p_gen.add_argument("--use_confidence_temp", action="store_true")
    p_gen.add_argument("--no_stop_on_eos", action="store_true")
    p_gen.add_argument("--no_kv_cache", action="store_true",
                       help="Disable the KV cache and recompute attention over the full "
                            "context every step (slower; only useful for debugging/comparison).")
    p_gen.add_argument("--seed", type=int, default=None)

    # eval
    p_eval = sub.add_parser("eval", help="Evaluate a checkpoint on val data")
    p_eval.add_argument("--ckpt", type=str, required=True)
    p_eval.add_argument("--data_dir", type=str, required=True)
    p_eval.add_argument("--device", type=str, default="auto")
    p_eval.add_argument("--dtype", type=str, default="auto", choices=["auto", "float32", "bfloat16", "float16"])
    p_eval.add_argument("--eval_iters", type=int, default=50)

    args = parser.parse_args()

    if args.command == "prepare":
        prepare_data(args.data_path, args.out_dir, args.val_fraction, args.seed)
    elif args.command == "train":
        cmd_train(args)
    elif args.command == "generate":
        cmd_generate(args)
    elif args.command == "eval":
        cmd_eval(args)
    else:
        parser.print_help()
        sys.exit(1)


if __name__ == "__main__":
    main()

Overwriting tensegrity_lm.py


## Write `tensegrity_nb_lib.py`


In [5]:
%%writefile tensegrity_nb_lib.py
"""
tensegrity_nb_lib.py — extension utilities for the TensegrityLM notebook.

These functions are designed to be dropped into notebook cells. They build on
top of tensegrity_lm.py (TensegrityConfig, TensegrityLM, Block, SwiGLU,
ffn_hidden_dim, get_batch, evaluate, save_checkpoint,
load_checkpoint, capture_rng_state, restore_rng_state, get_lr, pick_dtype).
"""

import os
import json
import dataclasses
import time
import numpy as np
import torch

from tensegrity_lm import (
    TensegrityConfig, TensegrityLM, ByteTokenizer,
    ffn_hidden_dim, get_batch, evaluate, save_checkpoint, load_checkpoint,
    capture_rng_state, restore_rng_state, get_lr, get_lr_wsd, pick_dtype, _get_memmap,
    MODEL_PRESETS, get_preset_config,
)


# =============================================================================
# Named configuration presets (notebook convenience wrappers)
# =============================================================================

def base15m_config(**overrides) -> TensegrityConfig:
    """
    ~15M-parameter reference TensegrityConfig for notebook use (see
    BASE15M_CONFIG_KWARGS / MODEL_PRESETS["base15m"] in tensegrity_lm.py, and
    test_presets.py which guards its param count against drift).
    Pass field overrides as kwargs, e.g. base15m_config(dropout=0.1).
    """
    return get_preset_config("base15m", **overrides)


# =============================================================================
# Multi-dataset ("replay") batching
# =============================================================================

def get_batch_mixed(data_dirs, weights, split, batch_size, block_size, device):
    """
    Like get_batch, but each row of the batch is sampled from one of several
    prepared data directories, chosen according to `weights` (normalized).
    Used to mix "old" and "new" corpora during continual training so the model
    keeps seeing examples of what it already learned (replay-based retention).
    """
    w = np.asarray(weights, dtype=np.float64)
    w = w / w.sum()

    arrs = []
    for d in data_dirs:
        path = os.path.join(d, f"{split}.bin")
        arr = _get_memmap(path)
        if len(arr) <= block_size + 1:
            raise ValueError(
                f"{path} has only {len(arr)} tokens, need > block_size+1 ({block_size + 1})."
            )
        arrs.append(arr)

    # Per-row RNG draws stay a Python loop (each row's max_start depends on which corpus
    # was chosen for it, and np.random must be consumed in this exact row-by-row order to
    # stay reproducible / bit-identical to the pre-vectorization implementation for a
    # given seed). What actually gets vectorized is the expensive part: gathering the
    # sampled windows out of the memmap-backed arrays. Rows are grouped by source corpus
    # (at most len(data_dirs) groups, vs. batch_size individual slice+append+stack calls)
    # and each group is gathered in one fancy-indexed batch instead of a per-row loop.
    choices = np.random.choice(len(data_dirs), size=batch_size, p=w)
    starts = np.empty(batch_size, dtype=np.int64)
    for row in range(batch_size):
        arr = arrs[choices[row]]
        max_start = len(arr) - block_size - 1
        starts[row] = np.random.randint(0, max_start)

    offsets = np.arange(block_size + 1)
    x = np.empty((batch_size, block_size), dtype=np.int64)
    y = np.empty((batch_size, block_size), dtype=np.int64)
    for d_idx in range(len(data_dirs)):
        rows = np.nonzero(choices == d_idx)[0]
        if len(rows) == 0:
            continue
        idx = starts[rows][:, None] + offsets[None, :]  # (len(rows), block_size + 1)
        arr = arrs[d_idx]
        xy = arr[idx].astype(np.int64)
        x[rows] = xy[:, :-1]
        y[rows] = xy[:, 1:]

    x_t = torch.from_numpy(x)
    y_t = torch.from_numpy(y)
    # startswith, not ==: "cuda:0"/"cuda:1" etc. (common on multi-GPU setups) previously
    # fell through to the no-pin-memory branch silently. Matches the same fix already
    # applied to get_batch in tensegrity_lm.py.
    if device.startswith("cuda"):
        x_t = x_t.pin_memory().to(device, non_blocking=True)
        y_t = y_t.pin_memory().to(device, non_blocking=True)
    else:
        x_t = x_t.to(device)
        y_t = y_t.to(device)
    return x_t, y_t


# =============================================================================
# Net2Net-style "wider FFN" growth (function-preserving at the moment of growth)
# =============================================================================

@torch.no_grad()
def widen_swiglu(old_ffn, new_ffn, old_hidden, new_hidden, bias):
    """
    Copy an old SwiGLU module's weights into a wider new SwiGLU module.

    - w_gate / w_up: old rows [0:old_hidden] are copied; the new_ffn's own
      (already randomly initialized) rows [old_hidden:new_hidden] are left as-is.
    - w_down: old columns [0:old_hidden] are copied; new columns
      [old_hidden:new_hidden] are zeroed.

    Because w_down's new input columns are zero, the new hidden units
    contribute exactly zero to the output regardless of what w_gate/w_up
    compute for them -> the FFN's output is IDENTICAL to the old FFN's output
    for any input, at the moment of growth. The new units only start
    influencing the output once gradients move w_down's new columns away
    from zero during continued training.
    """
    assert new_hidden >= old_hidden, "widen_swiglu only supports growing (new_hidden >= old_hidden)"

    new_ffn.w_gate.weight.data[:old_hidden, :].copy_(old_ffn.w_gate.weight.data)
    new_ffn.w_up.weight.data[:old_hidden, :].copy_(old_ffn.w_up.weight.data)

    new_ffn.w_down.weight.data[:, :old_hidden].copy_(old_ffn.w_down.weight.data)
    new_ffn.w_down.weight.data[:, old_hidden:].zero_()

    if bias:
        new_ffn.w_gate.bias.data[:old_hidden].copy_(old_ffn.w_gate.bias.data)
        new_ffn.w_up.bias.data[:old_hidden].copy_(old_ffn.w_up.bias.data)
        new_ffn.w_down.bias.data.copy_(old_ffn.w_down.bias.data)


@torch.no_grad()
def transplant_block(old_block, new_block, cfg_old, cfg_new):
    """
    Copy everything from old_block into new_block. Attention, norms, and
    residual gains are copied verbatim (their shapes never change under depth/FFN-only
    growth modes). The FFN is copied verbatim if its hidden_dim is
    unchanged, or widened via widen_swiglu if cfg_new.ffn_mult/ffn_align
    produced a larger hidden_dim. When cfg_new.n_embd > cfg_old.n_embd, embedding
    columns and projection features are widened with zero-padding.
    """
    old_D = cfg_old.n_embd
    new_D = cfg_new.n_embd

    if old_D == new_D:
        # --- Norms ---
        new_block.ln1.weight.data.copy_(old_block.ln1.weight.data)
        new_block.ln2.weight.data.copy_(old_block.ln2.weight.data)

        # --- Attention (shape-invariant under our depth/FFN growth modes) ---
        for name in ["q_proj", "k_proj", "v_proj", "o_proj"]:
            old_lin = getattr(old_block.attn, name)
            new_lin = getattr(new_block.attn, name)
            new_lin.weight.data.copy_(old_lin.weight.data)
            if cfg_old.bias:
                new_lin.bias.data.copy_(old_lin.bias.data)

        # --- Q/K RMSNorm ---
        if getattr(old_block.attn, "q_norm", None) is not None and getattr(new_block.attn, "q_norm", None) is not None:
            new_block.attn.q_norm.weight.data.copy_(old_block.attn.q_norm.weight.data)
            new_block.attn.k_norm.weight.data.copy_(old_block.attn.k_norm.weight.data)

        # --- Residual gains ---
        new_block.attn_gain.data.copy_(old_block.attn_gain.data)
        new_block.ffn_gain.data.copy_(old_block.ffn_gain.data)

        # --- FFN ---
        old_hidden = ffn_hidden_dim(cfg_old.n_embd, cfg_old.ffn_mult, cfg_old.ffn_align)
        new_hidden = ffn_hidden_dim(cfg_new.n_embd, cfg_new.ffn_mult, cfg_new.ffn_align)

        if old_hidden == new_hidden:
            new_block.ffn.w_gate.weight.data.copy_(old_block.ffn.w_gate.weight.data)
            new_block.ffn.w_up.weight.data.copy_(old_block.ffn.w_up.weight.data)
            new_block.ffn.w_down.weight.data.copy_(old_block.ffn.w_down.weight.data)
            if cfg_old.bias:
                new_block.ffn.w_gate.bias.data.copy_(old_block.ffn.w_gate.bias.data)
                new_block.ffn.w_up.bias.data.copy_(old_block.ffn.w_up.bias.data)
                new_block.ffn.w_down.bias.data.copy_(old_block.ffn.w_down.bias.data)
        else:
            widen_swiglu(old_block.ffn, new_block.ffn, old_hidden, new_hidden, cfg_old.bias)
    else:
        # --- Embedding width growth (n_embd) ---
        scale = (old_D / new_D) ** 0.5

        # RMSNorms: scale weight vector by sqrt(old_D / new_D) to adjust for RMS dimension
        # change. That alone is only exact when eps == 0: RMSNorm computes
        # rsqrt(mean(x^2) + eps), and zero-padding x from old_D to new_D shrinks
        # mean(x^2) by old_D/new_D while eps stays fixed, so the two terms no longer
        # scale together. Scaling eps by the same old_D/new_D factor keeps the ratio
        # eps/mean(x^2) identical to the old RMSNorm's, making the growth exact again
        # (not just approximately so) for every input, not only ones where eps is
        # negligible relative to mean(x^2).
        new_block.ln1.weight.data[:old_D].copy_(old_block.ln1.weight.data * scale)
        new_block.ln2.weight.data[:old_D].copy_(old_block.ln2.weight.data * scale)
        new_block.ln1.eps = old_block.ln1.eps * (scale ** 2)
        new_block.ln2.eps = old_block.ln2.eps * (scale ** 2)

        # Residual gains: when embed_scale is True, initial embeddings scale with
        # sqrt(new_D / old_D), so residual branch gains must scale by the same factor
        # to preserve exact relative branch balance and function preservation.
        gain_scale = (new_D / old_D) ** 0.5 if getattr(cfg_old, "embed_scale", False) else 1.0
        new_block.attn_gain.data.copy_(old_block.attn_gain.data * gain_scale)
        new_block.ffn_gain.data.copy_(old_block.ffn_gain.data * gain_scale)

        # Attention projections
        old_head_dim = cfg_old.n_embd // cfg_old.n_head
        new_head_dim = cfg_new.n_embd // cfg_new.n_head

        new_block.attn.q_proj.weight.data.zero_()
        new_block.attn.k_proj.weight.data.zero_()
        new_block.attn.v_proj.weight.data.zero_()
        new_block.attn.o_proj.weight.data.zero_()
        if cfg_old.bias:
            new_block.attn.q_proj.bias.data.zero_()
            new_block.attn.k_proj.bias.data.zero_()
            new_block.attn.v_proj.bias.data.zero_()
            new_block.attn.o_proj.bias.data.zero_()

        # grow_model() only ever calls transplant_block with a head_dim-preserving
        # (new_n_embd, new_n_head) pair -- see its assertion and the comment there.
        # RoPE's rotation frequencies are themselves a function of head_dim
        # (precompute_rope's inv_freq depends on head_dim, not just n_embd), so a
        # head_dim change would rotate every *existing* channel differently, not just
        # add new ones; there is no per-head partial-copy that stays exact under that.
        assert old_head_dim == new_head_dim, (
            "transplant_block's n_embd-growth path requires head_dim to stay fixed "
            f"(got old={old_head_dim}, new={new_head_dim}); this should be unreachable "
            "given grow_model()'s own assertion."
        )
        old_q_out = cfg_old.n_head * old_head_dim
        old_kv_out = cfg_old.n_kv_head * old_head_dim
        new_block.attn.q_proj.weight.data[:old_q_out, :old_D].copy_(old_block.attn.q_proj.weight.data)
        new_block.attn.k_proj.weight.data[:old_kv_out, :old_D].copy_(old_block.attn.k_proj.weight.data)
        new_block.attn.v_proj.weight.data[:old_kv_out, :old_D].copy_(old_block.attn.v_proj.weight.data)
        new_block.attn.o_proj.weight.data[:old_D, :old_q_out].copy_(old_block.attn.o_proj.weight.data)
        if cfg_old.bias:
            new_block.attn.q_proj.bias.data[:old_q_out].copy_(old_block.attn.q_proj.bias.data)
            new_block.attn.k_proj.bias.data[:old_kv_out].copy_(old_block.attn.k_proj.bias.data)
            new_block.attn.v_proj.bias.data[:old_kv_out].copy_(old_block.attn.v_proj.bias.data)
            new_block.attn.o_proj.bias.data[:old_D].copy_(old_block.attn.o_proj.bias.data)

        # Q/K RMSNorm: head_dim is fixed, so q_norm and k_norm weights are directly copied
        if getattr(old_block.attn, "q_norm", None) is not None and getattr(new_block.attn, "q_norm", None) is not None:
            new_block.attn.q_norm.weight.data.copy_(old_block.attn.q_norm.weight.data)
            new_block.attn.k_norm.weight.data.copy_(old_block.attn.k_norm.weight.data)

        # --- FFN ---
        old_hidden = ffn_hidden_dim(cfg_old.n_embd, cfg_old.ffn_mult, cfg_old.ffn_align)
        new_hidden = ffn_hidden_dim(cfg_new.n_embd, cfg_new.ffn_mult, cfg_new.ffn_align)

        def transplant_swiglu_embd(old_swiglu, new_swiglu):
            new_swiglu.w_gate.weight.data.zero_()
            new_swiglu.w_gate.weight.data[:old_hidden, :old_D].copy_(old_swiglu.w_gate.weight.data)
            new_swiglu.w_up.weight.data.zero_()
            new_swiglu.w_up.weight.data[:old_hidden, :old_D].copy_(old_swiglu.w_up.weight.data)
            new_swiglu.w_down.weight.data.zero_()
            new_swiglu.w_down.weight.data[:old_D, :old_hidden].copy_(old_swiglu.w_down.weight.data)

            if cfg_old.bias:
                new_swiglu.w_gate.bias.data[:old_hidden].copy_(old_swiglu.w_gate.bias.data)
                new_swiglu.w_up.bias.data[:old_hidden].copy_(old_swiglu.w_up.bias.data)
                new_swiglu.w_down.bias.data[:old_D].copy_(old_swiglu.w_down.bias.data)

        transplant_swiglu_embd(old_block.ffn, new_block.ffn)


_GROWTH_LOCKED_FIELDS = [
    "vocab_size", "block_size", "n_embd", "n_head", "n_kv_head",
    "sliding_window", "global_every",
    "bias", "dropout", "init_std", "homeo_target", "homeo_weight",
    "use_confidence_head", "confidence_weight",
    "use_qk_norm", "z_loss_weight", "logit_soft_cap", "embed_scale",
]


@torch.no_grad()
def grow_model(old_model, old_cfg, new_n_layer=None, ffn_growth_mult=None,
               new_n_embd=None, new_n_head=None, new_n_kv_head=None, device=None):
    """
    Create a strictly more capable model that, at the moment of growth,
    computes EXACTLY the same function as old_model (verified by the equality
    check in check_function_preserved).

    Three independent growth axes are supported, and any combination may be used:

      - new_n_layer:    append (new_n_layer - old_cfg.n_layer) new transformer
                        blocks at the end of the stack. New blocks have their
                        residual gains initialized to exactly 0, so they are
                        a no-op at init: x = x + 0*attn_out = x.
      - ffn_growth_mult: widen every block's FFN hidden dimension by this
                        factor (Net2Net "wider net"). New hidden units
                        contribute exactly 0 to the residual stream at init
                        because w_down's new input columns are zeroed.
      - new_n_embd / new_n_head: widen the residual stream dimension (n_embd).
                        Embedding columns, RMSNorm weights, and attention projections
                        are widened. By default, head_dim is kept fixed and n_head
                        (and n_kv_head proportionally for GQA) grows with n_embd,
                        preserving RoPE position frequencies and attention distribution.

    vocab_size, block_size, sliding_window, global_every, bias, and
    homeostasis/confidence hyperparameters are NOT changed by this function.
    """
    if device is None:
        device = next(old_model.parameters()).device

    new_cfg_kwargs = dataclasses.asdict(old_cfg)

    if new_n_layer is not None:
        assert new_n_layer >= old_cfg.n_layer, "new_n_layer must be >= old_cfg.n_layer"
        new_cfg_kwargs["n_layer"] = new_n_layer
        if new_n_layer != old_cfg.n_layer:
            # let TensegrityConfig.__post_init__ recompute gain_init for the new depth;
            # this only affects the *new* blocks (old blocks' gains are copied below).
            new_cfg_kwargs["gain_init"] = None

    if ffn_growth_mult is not None:
        assert ffn_growth_mult >= 1.0, "ffn_growth_mult must be >= 1.0"
        new_cfg_kwargs["ffn_mult"] = old_cfg.ffn_mult * ffn_growth_mult

    if new_n_embd is not None or new_n_head is not None:
        old_head_dim = old_cfg.n_embd // old_cfg.n_head
        if new_n_embd is not None:
            assert new_n_embd >= old_cfg.n_embd, "new_n_embd must be >= old_cfg.n_embd"
            new_cfg_kwargs["n_embd"] = new_n_embd
        else:
            assert new_n_head >= old_cfg.n_head, "new_n_head must be >= old_cfg.n_head"
            new_n_embd = new_n_head * old_head_dim
            new_cfg_kwargs["n_embd"] = new_n_embd

        # head_dim must stay fixed: precompute_rope's inv_freq is a function of head_dim
        # itself, so changing it would change the RoPE rotation angle applied to every
        # *existing* channel, not just add new ones -- there is no way to keep old
        # channels' attention scores exact under a head_dim change. Earlier revisions of
        # this function had a "custom head_dim" fallback for new_n_embd values that
        # don't divide old_head_dim evenly; it looked like it worked (fresh random-init
        # models pass loose atol checks) but was never actually exact -- verified to
        # diverge by ~0.08 max logit difference on a realistically-trained model, well
        # outside any reasonable tolerance. Only the fixed-head_dim convention is exact.
        if new_n_embd % old_head_dim != 0:
            raise ValueError(
                f"new_n_embd ({new_n_embd}) is not a multiple of the existing head_dim "
                f"({old_head_dim} = old n_embd {old_cfg.n_embd} // old n_head {old_cfg.n_head}). "
                f"grow_model only supports function-preserving growth that keeps head_dim "
                f"fixed (adding more heads, not widening each head) -- pick a new_n_embd "
                f"that's a multiple of {old_head_dim}, e.g. "
                f"{old_head_dim * (new_n_embd // old_head_dim + 1)}."
            )
        implied_n_head = new_n_embd // old_head_dim
        if new_n_head is None:
            new_n_head = implied_n_head
        elif new_n_head != implied_n_head:
            raise ValueError(
                f"new_n_head={new_n_head} would change head_dim from {old_head_dim} to "
                f"{new_n_embd // new_n_head} (not function-preserving under RoPE -- see "
                f"above). For new_n_embd={new_n_embd}, the only head_dim-preserving choice "
                f"is new_n_head={implied_n_head}; omit new_n_head to have it inferred."
            )
        assert new_n_head >= old_cfg.n_head, "new_n_head must be >= old_cfg.n_head"
        new_cfg_kwargs["n_head"] = new_n_head

        if new_n_kv_head is None:
            gqa_ratio = old_cfg.n_head // old_cfg.n_kv_head
            new_n_kv_head = max(1, new_n_head // gqa_ratio)
        new_cfg_kwargs["n_kv_head"] = new_n_kv_head

    new_cfg = TensegrityConfig(**new_cfg_kwargs)

    # Sanity: confirm none of the "locked" fields drifted.
    locked_fields = list(_GROWTH_LOCKED_FIELDS)
    if new_n_embd is not None or new_n_head is not None:
        for f in ["n_embd", "n_head", "n_kv_head"]:
            if f in locked_fields:
                locked_fields.remove(f)

    for field_name in locked_fields:
        old_val = getattr(old_cfg, field_name)
        new_val = getattr(new_cfg, field_name)
        assert old_val == new_val, (
            f"grow_model must not change '{field_name}' "
            f"(old={old_val!r}, new={new_val!r}); this would break function preservation."
        )

    old_hidden = ffn_hidden_dim(old_cfg.n_embd, old_cfg.ffn_mult, old_cfg.ffn_align)
    new_hidden = ffn_hidden_dim(new_cfg.n_embd, new_cfg.ffn_mult, new_cfg.ffn_align)
    if new_hidden < old_hidden:
        raise ValueError(
            f"Resulting ffn hidden_dim ({new_hidden}) is smaller than the current one "
            f"({old_hidden}). Increase ffn_growth_mult or leave it as None."
        )

    new_model = TensegrityLM(new_cfg)
    new_model.to(device)
    old_model.to(device)

    # --- Global / shared modules ---
    if old_cfg.n_embd != new_cfg.n_embd:
        scale = (old_cfg.n_embd / new_cfg.n_embd) ** 0.5
        new_model.token_emb.weight.data.zero_()
        new_model.token_emb.weight.data[:, :old_cfg.n_embd].copy_(old_model.token_emb.weight.data)
        new_model.ln_f.weight.data[:old_cfg.n_embd].copy_(old_model.ln_f.weight.data * scale)
        # See transplant_block's matching comment: eps must scale by old_D/new_D too,
        # not just the weight vector, for RMSNorm growth to be exact rather than approximate.
        new_model.ln_f.eps = old_model.ln_f.eps * (scale ** 2)
        if old_cfg.use_confidence_head and new_cfg.use_confidence_head:
            new_model.confidence_head.weight.data.zero_()
            new_model.confidence_head.weight.data[:, :old_cfg.n_embd].copy_(old_model.confidence_head.weight.data)
            new_model.confidence_head.bias.data.copy_(old_model.confidence_head.bias.data)
    else:
        new_model.token_emb.weight.data.copy_(old_model.token_emb.weight.data)  # also lm_head (tied)
        new_model.ln_f.weight.data.copy_(old_model.ln_f.weight.data)
        if old_cfg.use_confidence_head and new_cfg.use_confidence_head:
            new_model.confidence_head.weight.data.copy_(old_model.confidence_head.weight.data)
            new_model.confidence_head.bias.data.copy_(old_model.confidence_head.bias.data)

    # --- Existing blocks: copy / widen ---
    for i in range(old_cfg.n_layer):
        transplant_block(old_model.blocks[i], new_model.blocks[i], old_cfg, new_cfg)

    # --- New blocks (depth growth only): zero residual gains -> exact no-op ---
    for i in range(old_cfg.n_layer, new_cfg.n_layer):
        new_model.blocks[i].attn_gain.data.zero_()
        new_model.blocks[i].ffn_gain.data.zero_()

    return new_model, new_cfg


@torch.no_grad()
def check_function_preserved(old_model, new_model, cfg, device, n_tokens=48, batch=2, atol=1e-5):
    """
    Feed identical random byte sequences through old_model and new_model
    (both in eval mode, fp32, no autocast) and confirm the logits match.
    This is the empirical proof that grow_model() retained everything the
    old model knew.
    """
    if old_model.cfg.n_embd != new_model.cfg.n_embd and atol == 1e-5:
        atol = 1e-3
    was_training_old = old_model.training
    was_training_new = new_model.training
    old_model.eval()
    new_model.eval()
    # TF32 silently truncates matmul mantissas on Ampere+ GPUs and is NOT part of
    # autocast -- left enabled (its default), it can push "exact" fp32 growth checks
    # over typical atol thresholds purely from matmul rounding, unrelated to any real
    # function-preservation bug. Disable it for the duration of this check only.
    prev_matmul_tf32 = torch.backends.cuda.matmul.allow_tf32
    prev_cudnn_tf32 = torch.backends.cudnn.allow_tf32
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
    try:
        torch.manual_seed(0)
        x = torch.randint(0, cfg.vocab_size, (batch, min(n_tokens, cfg.block_size)), device=device)
        old_logits, _, _ = old_model(x, targets=None)
        new_logits, _, _ = new_model(x, targets=None)
        max_diff = (old_logits - new_logits).abs().max().item()
        ok = max_diff < atol
        print(f"[grow_model] max |old_logits - new_logits| = {max_diff:.3e} "
              f"({'PASS' if ok else 'FAIL'}, atol={atol:.1e})")
        if not ok:
            raise AssertionError(
                "grow_model() did not preserve the model's function. "
                "Do not continue training on this model until this is fixed."
            )
        return max_diff
    finally:
        torch.backends.cuda.matmul.allow_tf32 = prev_matmul_tf32
        torch.backends.cudnn.allow_tf32 = prev_cudnn_tf32
        if was_training_old:
            old_model.train()
        if was_training_new:
            new_model.train()


# =============================================================================
# Elastic Weight Consolidation (EWC) — "retain information"
# =============================================================================

def compute_fisher_diagonal(model, cfg, data_dir, device, device_type, dtype="auto",
                             n_batches=50, batch_size=8, chunk_size=None,
                             fisher_dtype="float32"):
    """
    Estimate the diagonal of the Fisher information matrix for every
    parameter, using the cross-entropy loss on `n_batches` random batches
    drawn from data_dir's train split. Larger values mean "this weight
    matters more for this dataset; don't let it move much."

    chunk_size: if given, process n_batches in groups of this size, explicitly
        freeing intermediate CUDA tensors (del + torch.cuda.empty_cache()) between
        chunks. Purely a peak-memory knob for constrained hardware -- the running
        fisher accumulator itself always stays resident throughout (that's the whole
        point of it being a running sum), so this only bounds the *transient* memory
        used by each batch's forward/backward, not the final result's footprint.
        Default None preserves prior behavior exactly (one un-chunked loop).
    fisher_dtype: "float32" (default, unchanged) or "float16" -- dtype of the
        *returned* fisher dict. Accumulation during the loop always happens in fp32
        regardless of this setting (summing many fp16 squared-gradient terms would
        degrade precision compounding over n_batches); only the final returned tensors
        are cast, roughly halving the dict's memory footprint for continual-learning
        workflows that keep it resident across a whole training run. Halving here is
        the accuracy tradeoff being documented, not a bug -- fp16 can't represent the
        full dynamic range of squared gradients as precisely as fp32.

    Returns a dict {param_name: fisher_tensor} (same shapes as the model's
    current parameters) and a matching snapshot {param_name: value_tensor}
    of the parameters at the time of computation.
    """
    assert fisher_dtype in ("float32", "float16"), (
        f"fisher_dtype must be 'float32' or 'float16', got {fisher_dtype!r}"
    )
    assert n_batches >= 1, "n_batches must be >= 1"
    amp_dtype = pick_dtype(dtype, device_type)
    use_amp = amp_dtype in (torch.float16, torch.bfloat16) and device_type in ("cuda", "cpu")

    was_training = model.training
    model.eval()

    fisher = {n: torch.zeros_like(p, dtype=torch.float32) for n, p in model.named_parameters()
              if p.requires_grad}
    old_params = {n: p.detach().clone() for n, p in model.named_parameters() if p.requires_grad}

    chunk_size = chunk_size or n_batches
    batches_done = 0
    while batches_done < n_batches:
        this_chunk = min(chunk_size, n_batches - batches_done)
        for _ in range(this_chunk):
            x, y = get_batch(data_dir, "train", batch_size, cfg.block_size, device)
            model.zero_grad(set_to_none=True)
            with torch.amp.autocast(device_type=device_type if device_type != "mps" else "cpu",
                                      dtype=amp_dtype, enabled=use_amp):
                _, losses, _ = model(x, y)
                loss = losses["ce_loss"]
            loss.backward()
            for n, p in model.named_parameters():
                if p.grad is not None:
                    fisher[n] += p.grad.detach().float().pow(2)
            del x, y, losses, loss
        batches_done += this_chunk
        model.zero_grad(set_to_none=True)
        if device_type == "cuda":
            torch.cuda.empty_cache()

    for n in fisher:
        fisher[n] /= float(n_batches)

    model.zero_grad(set_to_none=True)
    if was_training:
        model.train()

    if fisher_dtype == "float16":
        fisher = {n: t.half() for n, t in fisher.items()}

    return fisher, old_params


@torch.no_grad()
def expand_reference_state(old_fisher, old_params, new_model):
    """
    Re-map an (old_fisher, old_params) pair computed on the pre-growth model
    onto the post-growth model's parameter shapes:

      - Parameters whose shape is unchanged (almost everything: attention,
        norms, gains, embeddings, old FFN columns) are carried over directly.
      - Widened FFN tensors are zero-padded: the old region keeps its real
        Fisher value (and its old value, so EWC pulls it back there), while
        the newly-added region gets Fisher = 0 and old_value = its current
        (freshly initialized / post-grow) value, so EWC imposes NO penalty
        on the new capacity — it is free to specialize on new data.
      - Parameters that did not exist before growth (new transformer blocks)
        are absent from the returned dicts entirely -> also unconstrained.

    NOTE: For multi-stage growth (grow -> train -> grow again), you should
    recompute Fisher on the intermediate model after each training phase and
    pass that fresh Fisher to the next expand_reference_state call. Using the
    original pre-growth Fisher on a twice-grown model will under-constrain the
    first growth's new capacity.
    """
    if old_fisher is None or old_params is None:
        return {}, {}

    new_fisher = {}
    new_old_params = {}
    for name, p in new_model.named_parameters():
        if name not in old_fisher:
            continue
        of = old_fisher[name]
        op = old_params[name]
        if of.shape == p.shape:
            new_fisher[name] = of.to(p.device)
            new_old_params[name] = op.to(p.device)
        else:
            nf = torch.zeros_like(p, dtype=torch.float32)
            nop = p.detach().clone()
            slices = tuple(slice(0, min(s, p.shape[i])) for i, s in enumerate(of.shape))
            nf[slices] = of[slices].to(p.device)
            nop[slices] = op[slices].to(p.device)
            new_fisher[name] = nf
            new_old_params[name] = nop
    return new_fisher, new_old_params


def ewc_penalty(model, fisher, old_params):
    """sum_i fisher_i * (theta_i - theta_old_i)^2, over parameters present in `fisher`."""
    total = None
    for name, p in model.named_parameters():
        f = fisher.get(name)
        if f is None:
            continue
        term = (f * (p - old_params[name]).pow(2)).sum()
        total = term if total is None else total + term
    if total is None:
        return torch.zeros((), device=next(model.parameters()).device)
    return total


# =============================================================================
# Training loop (notebook-friendly: single device, no DDP/compile)
# =============================================================================

def run_training(model, optimizer, cfg, out_dir, device,
                  data_dirs, data_weights=None,
                  max_steps=1000, warmup_steps=50,
                  learning_rate=3e-4, min_lr=3e-5,
                  schedule="cosine", stable_steps=None, decay_steps=None,
                  grad_clip=1.0, batch_size=12, grad_accum_steps=1,
                  log_interval=10, eval_interval=100, eval_iters=20,
                  dtype="auto", start_step=0, best_val_loss=float("inf"),
                  ewc_fisher=None, ewc_old_params=None, ewc_lambda=0.0,
                  init_from="scratch"):
    """
    Train `model` for steps [start_step, max_steps). Checkpoints
    (ckpt_latest.pt / ckpt_best.pt) and a train_log.jsonl are written to
    out_dir. Validation is always measured against data_dirs[0].

    data_dirs: str (single prepared corpus) or list of prepared corpus dirs.
    data_weights: sampling weights over data_dirs (only used if len > 1);
                   normalized internally. Use this to "replay" an old corpus
                   while training on a new one.
    ewc_*: outputs of compute_fisher_diagonal / expand_reference_state, plus
           a weight ewc_lambda. Set ewc_lambda=0.0 (default) to disable. Not
           affected by init_from="resume" -- EWC anchors are not part of the
           checkpoint schema and must still be supplied fresh by the caller
           on every call (e.g. re-run compute_fisher_diagonal /
           expand_reference_state before resuming a continual-training run).
    init_from: "scratch" (default; preserves prior behavior exactly, no
        checkpoint is read) or "resume". "resume" looks for
        {out_dir}/ckpt_latest.pt; if found, loads the checkpoint's model /
        optimizer / GradScaler / RNG state IN PLACE into the `model` and
        `optimizer` objects you passed in, and overrides start_step /
        best_val_loss with the checkpoint's values. If no checkpoint exists
        at out_dir yet, falls back to the scratch behavior using whatever
        start_step/best_val_loss you passed (with a printed notice).

        IMPORTANT: unlike tensegrity_lm.py's cmd_train, this function does
        not construct the model itself -- it can only load weights into the
        model/optimizer you already built. If the checkpoint at out_dir was
        saved under a different TensegrityConfig than `cfg`, resuming would
        load mismatched weights into a differently-shaped model, sometimes
        with a clear shape-mismatch error and sometimes (if parameter counts
        happen to coincide) with no error but silently wrong weights. This
        is checked explicitly below and raises before any loading happens.

    schedule: "cosine" (default; preserves prior behavior exactly) or "wsd" (warmup-
        stable-decay, see get_lr_wsd in tensegrity_lm.py). "wsd" requires stable_steps
        and decay_steps; warmup_steps + stable_steps + decay_steps need not equal
        max_steps (the schedule just holds at min_lr past decay_end, same convention
        as get_lr past max_steps).

        Either schedule is applied as a per-group MULTIPLIER, not an absolute value:
        each optimizer.param_groups[i]'s "lr" at construction time (its "base lr") is
        captured once before training starts, and every step's schedule value scales
        that base proportionally. For a plain AdamW optimizer (every group built with
        the same lr=learning_rate) this is mathematically identical to the old
        "overwrite every group to the same absolute lr" behavior. It's what makes a
        Muon optimizer (whose groups intentionally start at different base LRs --
        e.g. 0.02 for the Muon group vs. learning_rate for its AdamW groups, see
        TensegrityLM.configure_optimizer(use_muon=True)) work correctly here: without
        this, every group would get stomped to the same absolute value every step,
        silently breaking Muon's intended LR scale.

    Returns (final_step, best_val_loss).
    """
    assert init_from in ("scratch", "resume"), (
        f"init_from must be 'scratch' or 'resume', got {init_from!r}"
    )
    assert schedule in ("cosine", "wsd"), f"schedule must be 'cosine' or 'wsd', got {schedule!r}"
    if schedule == "wsd":
        assert stable_steps is not None and decay_steps is not None, (
            "schedule='wsd' requires stable_steps and decay_steps"
        )
    assert grad_accum_steps >= 1, (
        f"grad_accum_steps must be >= 1, got {grad_accum_steps!r}"
    )

    os.makedirs(out_dir, exist_ok=True)
    device_type = "cuda" if "cuda" in device else ("mps" if device == "mps" else "cpu")
    amp_dtype = pick_dtype(dtype, device_type)
    use_amp = amp_dtype in (torch.float16, torch.bfloat16) and device_type in ("cuda", "cpu")
    # Constructed before any resume-load below (same fix as cmd_train in
    # tensegrity_lm.py): scaler must exist before scaler.load_state_dict can
    # be called on it.
    scaler = torch.amp.GradScaler(enabled=(amp_dtype == torch.float16 and device_type == "cuda"))

    if init_from == "resume":
        latest_path = os.path.join(out_dir, "ckpt_latest.pt")
        if os.path.exists(latest_path):
            ckpt_resume = load_checkpoint(latest_path, device)

            cur_cfg_dict = dataclasses.asdict(cfg)
            ckpt_cfg_dict = ckpt_resume["config"]
            assert cur_cfg_dict == ckpt_cfg_dict, (
                f"init_from='resume' found a checkpoint at {latest_path} whose saved "
                f"config does not match the `cfg` passed to run_training. Resuming "
                f"would load mismatched weights into a differently-shaped/configured "
                f"model. Reconstruct `model` with TensegrityConfig(**checkpoint['config']) "
                f"before calling run_training, or point `out_dir` at a fresh directory "
                f"for a from-scratch run."
            )

            model.load_state_dict(ckpt_resume["model_state_dict"])
            optimizer.load_state_dict(ckpt_resume["optimizer_state_dict"])
            restore_rng_state(ckpt_resume.get("rng_state"))
            if ckpt_resume.get("scaler_state_dict") is not None:
                scaler.load_state_dict(ckpt_resume["scaler_state_dict"])

            # Checkpoints are saved with the step number that was just completed
            # (see the save_checkpoint calls below, before `step += 1`), so resuming
            # must continue from the next step, not repeat the one already trained.
            start_step = ckpt_resume["step"] + 1
            best_val_loss = ckpt_resume["best_val_loss"]
            print(f"[resume] loaded checkpoint from {latest_path} at step={start_step}, "
                  f"best_val_loss={best_val_loss:.4f}")
        else:
            print(f"[resume] no checkpoint found at {latest_path}; starting from scratch "
                  f"(start_step={start_step}, best_val_loss={best_val_loss})")

    if isinstance(data_dirs, str):
        data_dirs = [data_dirs]
    if data_weights is None:
        data_weights = [1.0] * len(data_dirs)
    assert len(data_weights) == len(data_dirs)

    log_path = os.path.join(out_dir, "train_log.jsonl")
    log_file = open(log_path, "a")

    # Each group's own starting lr (set by configure_optimizer -- e.g. 0.02 for a Muon
    # group, learning_rate for AdamW groups) is the multiplier's base. For a plain
    # AdamW optimizer every group starts at the same lr=learning_rate, so this reduces
    # exactly to the old single-value behavior; see the schedule= docstring above.
    base_lrs = [g["lr"] for g in optimizer.param_groups]
    floor_ratio = (min_lr / learning_rate) if learning_rate != 0 else 0.0

    try:
        model.train()
        optimizer.zero_grad(set_to_none=True)
        step = start_step
        t0 = time.time()
        last_losses = None

        while step < max_steps:
            if schedule == "wsd":
                lr_mult = get_lr_wsd(step, warmup_steps, stable_steps, decay_steps,
                                      max_lr=1.0, min_lr=floor_ratio)
            else:
                lr_mult = get_lr(step, warmup_steps, max_steps, max_lr=1.0, min_lr=floor_ratio)
            for g, base in zip(optimizer.param_groups, base_lrs):
                g["lr"] = base * lr_mult

            accum_total = 0.0
            for _ in range(grad_accum_steps):
                if len(data_dirs) == 1:
                    x, y = get_batch(data_dirs[0], "train", batch_size, cfg.block_size, device)
                else:
                    x, y = get_batch_mixed(data_dirs, data_weights, "train", batch_size, cfg.block_size, device)

                with torch.amp.autocast(device_type=device_type if device_type != "mps" else "cpu",
                                          dtype=amp_dtype, enabled=use_amp):
                    logits, losses, _ = model(x, y)
                    total = losses["total_loss"]
                    if ewc_lambda > 0.0 and ewc_fisher is not None:
                        total = total + ewc_lambda * ewc_penalty(model, ewc_fisher, ewc_old_params)
                    scaled = total / grad_accum_steps

                if scaler.is_enabled():
                    scaler.scale(scaled).backward()
                else:
                    scaled.backward()

                accum_total += total.item() / grad_accum_steps
                last_losses = losses

            if grad_clip > 0.0:
                if scaler.is_enabled():
                    scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)

            if scaler.is_enabled():
                scaler.step(optimizer)
                scaler.update()
            else:
                optimizer.step()
            optimizer.zero_grad(set_to_none=True)

            if step % log_interval == 0:
                dt = time.time() - t0
                ce = last_losses["ce_loss"].item()
                homeo = last_losses["homeo_loss"].item()
                conf = last_losses.get("confidence_loss")
                conf_str = f" conf={conf.item():.4f}" if conf is not None else ""
                # optimizer.param_groups[0]'s actual current lr (not the raw 0..1 multiplier):
                # for a plain-AdamW optimizer this is the single lr every group shares, same
                # number the old log always showed. For Muon it's the Muon group's lr (its
                # own base_lrs[0] * lr_mult) -- lr_mult itself is also logged since it's the
                # one number that's directly comparable across every group regardless of base.
                lr0 = optimizer.param_groups[0]["lr"]
                print(f"step {step:6d} | lr {lr0:.2e} (mult {lr_mult:.3f}) | total {accum_total:.4f} "
                      f"(ce {ce:.4f} homeo {homeo:.4f}{conf_str}) | {dt:.1f}s")
                log_file.write(json.dumps({
                    "step": step, "lr": lr0, "lr_mult": lr_mult, "total_loss": accum_total,
                    "ce_loss": ce, "homeo_loss": homeo, "time": dt,
                }) + "\n")
                log_file.flush()

            if step > 0 and step % eval_interval == 0:
                val_loss = evaluate(model, data_dirs[0], cfg, device, device_type, amp_dtype, use_amp, eval_iters)
                print(f"step {step:6d} | val_loss[{os.path.basename(data_dirs[0])}] {val_loss:.4f}")
                log_file.write(json.dumps({"step": step, "val_loss": val_loss, "val_dir": data_dirs[0]}) + "\n")
                log_file.flush()

                rng_state = capture_rng_state()
                # BUGFIX: was `os.path.join(args.out_dir, ...)` and `raw_model` — neither
                # `args` nor `raw_model` exist in this function's scope (that's cmd_train's
                # vocabulary in tensegrity_lm.py, not run_training's). Caused a guaranteed
                # NameError the first time eval_interval was reached. Also now passes
                # scaler=scaler consistently at all four save points below, not just this one,
                # so GradScaler state (fp16 loss-scale) actually survives every checkpoint.
                save_checkpoint(os.path.join(out_dir, "ckpt_latest.pt"), model, optimizer, cfg, step, best_val_loss, rng_state, scaler=scaler)
                if val_loss < best_val_loss:
                    best_val_loss = val_loss
                    save_checkpoint(os.path.join(out_dir, "ckpt_best.pt"), model, optimizer, cfg, step, best_val_loss, rng_state, scaler=scaler)
                    print(f"step {step:6d} | new best val_loss {best_val_loss:.4f} -> ckpt_best.pt")

            step += 1

        val_loss = evaluate(model, data_dirs[0], cfg, device, device_type, amp_dtype, use_amp, eval_iters)
        rng_state = capture_rng_state()
        save_checkpoint(os.path.join(out_dir, "ckpt_latest.pt"), model, optimizer, cfg, step, best_val_loss, rng_state, scaler=scaler)
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            save_checkpoint(os.path.join(out_dir, "ckpt_best.pt"), model, optimizer, cfg, step, best_val_loss, rng_state, scaler=scaler)
        print(f"[train] done. step={step} final_val_loss[{os.path.basename(data_dirs[0])}]={val_loss:.4f} "
              f"best_val_loss={best_val_loss:.4f}")
        return step, best_val_loss
    finally:
        log_file.close()


# =============================================================================
# safetensors export / import
# =============================================================================

def export_to_safetensors(model, cfg, out_dir, name="model"):
    """
    Write {out_dir}/{name}.safetensors (weights only) and
    {out_dir}/{name}.config.json (TensegrityConfig, needed to reconstruct
    the model before loading the weights back in).

    token_emb.weight and lm_head.weight are tied (the same tensor); since
    safetensors cannot store two tensors that share memory, lm_head.weight
    is cloned into an independent copy before saving. Both keys are written
    (with identical values), so the file loads cleanly with either a tied or
    untied model.
    """
    from safetensors.torch import save_file

    os.makedirs(out_dir, exist_ok=True)
    state_dict = model.state_dict()

    tensors = {}
    for k, v in state_dict.items():
        t = v.detach().cpu().contiguous()
        tensors[k] = t

    # Break the tie: clone lm_head.weight so it doesn't alias token_emb.weight.
    if "lm_head.weight" in tensors and "token_emb.weight" in tensors:
        if tensors["lm_head.weight"].data_ptr() == tensors["token_emb.weight"].data_ptr():
            tensors["lm_head.weight"] = tensors["lm_head.weight"].clone()

    weights_path = os.path.join(out_dir, f"{name}.safetensors")
    save_file(tensors, weights_path, metadata={"format": "pt"})

    config_path = os.path.join(out_dir, f"{name}.config.json")
    with open(config_path, "w") as f:
        json.dump(dataclasses.asdict(cfg), f, indent=2)

    print(f"[export] wrote {weights_path}")
    print(f"[export] wrote {config_path}")
    return weights_path, config_path


def load_from_safetensors(weights_path, config_path, device="cpu"):
    """
    Reconstruct a TensegrityLM from a .safetensors weights file and its
    companion config.json (as written by export_to_safetensors).
    """
    from safetensors.torch import load_file

    with open(config_path) as f:
        cfg = TensegrityConfig(**json.load(f))

    model = TensegrityLM(cfg)
    state_dict = load_file(weights_path)
    model.load_state_dict(state_dict)
    # Re-establish weight tying: safetensors stores lm_head and token_emb as
    # independent tensors, so load_state_dict breaks the tie. Restore it here.
    model.lm_head.weight = model.token_emb.weight
    model.to(device)
    return model, cfg


# =============================================================================
# Confidence-head calibration
# =============================================================================

@torch.no_grad()
def evaluate_confidence_calibration(model, cfg, data_dir, device, device_type, dtype="auto",
                                     n_batches=20, batch_size=8, n_bins=10):
    """
    Measurement/tooling only -- does not change confidence_weight or the confidence
    loss formula (see TensegrityLM.forward()'s confidence_head branch).

    The confidence head predicts P(argmax prediction == target). This buckets that
    predicted probability into n_bins bins over [0, 1] using data_dir's val split, and
    for each bin computes the *empirical* accuracy (fraction of tokens in that bin whose
    argmax prediction was actually correct). A well-calibrated head has empirical
    accuracy close to the bin's mean predicted confidence in every bin -- e.g. tokens
    the head scored ~0.8 confident should be right ~80% of the time. Also reports
    Expected Calibration Error (ECE): the count-weighted average |accuracy - confidence|
    gap across bins, a single scalar summary (0 = perfectly calibrated).

    Returns {"bins": [{"bin_lo", "bin_hi", "count", "accuracy", "mean_confidence"}, ...],
             "ece": float, "n_tokens": int}.
    """
    assert model.confidence_head is not None, (
        "model has no confidence head (cfg.use_confidence_head=False); nothing to calibrate."
    )
    amp_dtype = pick_dtype(dtype, device_type)
    use_amp = amp_dtype in (torch.float16, torch.bfloat16) and device_type in ("cuda", "cpu")

    was_training = model.training
    model.eval()

    bin_edges = torch.linspace(0.0, 1.0, n_bins + 1)
    bin_correct = torch.zeros(n_bins)
    bin_conf_sum = torch.zeros(n_bins)
    bin_count = torch.zeros(n_bins)

    try:
        for _ in range(n_batches):
            x, y = get_batch(data_dir, "val", batch_size, cfg.block_size, device)
            with torch.amp.autocast(device_type=device_type if device_type != "mps" else "cpu",
                                      dtype=amp_dtype, enabled=use_amp):
                logits, _, confidence = model(x, targets=None)
            preds = logits.argmax(dim=-1)
            correct = (preds == y).float().reshape(-1).cpu()
            conf = confidence.float().reshape(-1).clamp(0.0, 1.0).cpu()

            # Right-closed bins except the last, which also absorbs conf == 1.0 exactly.
            bin_idx = torch.bucketize(conf, bin_edges[1:-1], right=False)
            bin_count += torch.bincount(bin_idx, minlength=n_bins).float()
            bin_correct += torch.bincount(bin_idx, weights=correct, minlength=n_bins).float()
            bin_conf_sum += torch.bincount(bin_idx, weights=conf, minlength=n_bins).float()
    finally:
        if was_training:
            model.train()

    total = bin_count.sum().item()
    rows = []
    ece = 0.0
    for b in range(n_bins):
        n = bin_count[b].item()
        if n == 0:
            acc, mean_conf = float("nan"), float("nan")
        else:
            acc = (bin_correct[b] / bin_count[b]).item()
            mean_conf = (bin_conf_sum[b] / bin_count[b]).item()
            ece += (n / total) * abs(acc - mean_conf)
        rows.append({
            "bin_lo": bin_edges[b].item(), "bin_hi": bin_edges[b + 1].item(),
            "count": int(n), "accuracy": acc, "mean_confidence": mean_conf,
        })

    print(f"{'bin':>14s} {'count':>8s} {'accuracy':>10s} {'mean_conf':>10s}")
    for row in rows:
        acc_str = f"{row['accuracy']:.4f}" if row["count"] > 0 else "n/a"
        conf_str = f"{row['mean_confidence']:.4f}" if row["count"] > 0 else "n/a"
        print(f"[{row['bin_lo']:.2f},{row['bin_hi']:.2f}) {row['count']:8d} "
              f"{acc_str:>10s} {conf_str:>10s}")
    print(f"Expected Calibration Error (ECE): {ece:.4f}  (n_tokens={int(total)})")

    return {"bins": rows, "ece": ece, "n_tokens": int(total)}


Overwriting tensegrity_nb_lib.py


## Imports, device, and working directories

In [6]:
import os
import sys
import time

import numpy as np
import torch

sys.path.insert(0, os.getcwd())

from tensegrity_lm import (
    TensegrityConfig, TensegrityLM, ByteTokenizer,
    prepare_data, get_batch, evaluate,
    save_checkpoint, load_checkpoint, capture_rng_state,
    get_lr, pick_device, pick_dtype,
    get_preset_config,  # added: needed by the base15m preset cell below
)
from tensegrity_nb_lib import (
    get_batch_mixed,
    grow_model, check_function_preserved,
    compute_fisher_diagonal, expand_reference_state, ewc_penalty,
    run_training, export_to_safetensors, load_from_safetensors,
)

SEED = 1337
torch.manual_seed(SEED)
np.random.seed(SEED)

DEVICE = pick_device("auto")
DEVICE_TYPE = "cuda" if "cuda" in DEVICE else ("mps" if DEVICE == "mps" else "cpu")
print(f"device = {DEVICE}  (device_type = {DEVICE_TYPE})")

ROOT = os.getcwd()
DATA_DIR = os.path.join(ROOT, "data")
# Was hardcoded to a Colab-only "/content/drive/..." path (os.path.join silently
# discards ROOT when the second arg is already absolute, so this broke outside
# Colab). Now a plain local dir like DATA_DIR/SAMPLE_DIR so it works on any machine.
# Colab disconnects/timeouts are the same failure mode as the local GPU driver
# crash that started this whole exercise (2026-08-08): an in-progress run dies and
# whatever wasn't checkpointed is gone. save_checkpoint runs every eval_interval, so
# pointing RUNS_DIR at mounted Drive (already mounted above, cell "drive.mount") means
# a disconnect loses at most one eval_interval's worth of steps, not the whole run.
# Falls back to a local dir automatically if Drive isn't mounted (e.g. running this
# notebook outside Colab).
if os.path.isdir("/kaggle/working"):
    RUNS_DIR = "/kaggle/working/tensegrity/nsfw002-001"
else:
    RUNS_DIR = os.path.join(ROOT, "runs")
SAMPLE_DIR = os.path.join(ROOT, "sample_data")
for _d in (DATA_DIR, RUNS_DIR, SAMPLE_DIR):
    os.makedirs(_d, exist_ok=True)

tok = ByteTokenizer()


device = cpu  (device_type = cpu)


## Clear data

In [ ]:
import shutil
import os

if os.path.exists(DATA_DIR):
    shutil.rmtree(DATA_DIR)
os.makedirs(DATA_DIR, exist_ok=True)
print(f"Cleared all contents in {DATA_DIR}")

### Prepare Parquet Data
This cell extracts text from the local parquet file in `/content/data` (using the `story` column identified in previous attempts) and runs the `prepare_data` pipeline for it.

In [ ]:
import pandas as pd
import os

# Source file identified from the environment
PARQUET_SRC = '/kaggle/input/datasets/damiendavison/notebook2a/train-combined.parquet'
DATA_PARQUET_OUT = os.path.join(DATA_DIR, "initial")
TXT_TEMP_PATH = os.path.join(SAMPLE_DIR, "extracted_stories.txt")

if os.path.exists(PARQUET_SRC):
    print(f"Reading parquet from {PARQUET_SRC}...")
    df_parquet = pd.read_parquet(PARQUET_SRC)

    # Based on the error log, the correct column is 'story'
    if 'story' in df_parquet.columns:
        print("Extracting 'story' column...")
        text_data = "\n\n".join(df_parquet['story'].dropna().astype(str).tolist())

        with open(TXT_TEMP_PATH, 'w', encoding='utf-8') as f:
            f.write(text_data)

        print(f"Saved text to {TXT_TEMP_PATH}. Preparing bin files...")
        prepare_data(TXT_TEMP_PATH, DATA_PARQUET_OUT, val_fraction=0.05, seed=SEED)
        print(f"Data prepared in: {DATA_PARQUET_OUT}")
    else:
        print(f"Error: 'story' column not found. Available: {df_parquet.columns.tolist()}")
else:
    print(f"File not found at {PARQUET_SRC}")

## Load custom data

In [ ]:
import os
import urllib.request

# Correct URL for RAW text content
TINYSTORIES_RAW_URL = "https://huggingface.co/datasets/roneneldan/TinyStories/resolve/main/TinyStories-train.txt"
TINYSTORIES_SLICE_BYTES = 420_000_000

CUSTOM_DATA_PATH = os.path.join(SAMPLE_DIR, "tinystories_slice_raw.txt")

print(f"Downloading {TINYSTORIES_SLICE_BYTES:,} bytes of RAW TinyStories text...")
req = urllib.request.Request(
    TINYSTORIES_RAW_URL, headers={"Range": f"bytes=0-{TINYSTORIES_SLICE_BYTES - 1}"}
)
with urllib.request.urlopen(req) as resp, open(CUSTOM_DATA_PATH, "wb") as f:
    f.write(resp.read())

print(f"Saved RAW slice to -> {CUSTOM_DATA_PATH} ({os.path.getsize(CUSTOM_DATA_PATH):,} bytes)")

# Re-prepare the data using the actual text content
DATA_INITIAL = os.path.join(DATA_DIR, "initial")
prepare_data(CUSTOM_DATA_PATH, DATA_INITIAL, val_fraction=0.02, seed=SEED)


## Load your own corpus (mixed in at low weight throughout training)

In [ ]:
import os
import urllib.request

# Direct URL to the custom corpus file
OWN_CORPUS_URL = "https://huggingface.co/datasets/roneneldan/TinyStories/resolve/main/TinyStories-valid.txt"
OWN_CORPUS_FILE = os.path.join(SAMPLE_DIR, "my_custom_corpus.txt")

if not os.path.exists(OWN_CORPUS_FILE):
    print(f"Downloading custom corpus from {OWN_CORPUS_URL}...")
    try:
        urllib.request.urlretrieve(OWN_CORPUS_URL, OWN_CORPUS_FILE)
        print(f"Saved -> {OWN_CORPUS_FILE}")
    except Exception as e:
        print(f"Failed to download: {e}")
else:
    print(f"Using cached {OWN_CORPUS_FILE}")

DATA_OWN = os.path.join(DATA_DIR, "own")
prepare_data(OWN_CORPUS_FILE, DATA_OWN, val_fraction=0.1, seed=SEED)

In [7]:
DATA_INITIAL = os.path.join(DATA_DIR, "initial")


## Model configuration

In [9]:
# Pareto-optimal 133M speed/intelligence configuration for <=14GB VRAM
# Balanced across parameter capacity, byte-level context length, and Tensor Core alignment.
cfg = TensegrityConfig(
    block_size=2048,             # Byte-level context: ~500 English words (8x over base15m)
    n_layer=12,                  # 12 transformer blocks for deep hierarchical representations
    n_embd=1024,                 # 1024 embedding dimension
    n_head=16,                   # 16 heads -> head_dim = 64 (Tensor Core native tile size)
    n_kv_head=4,                 # 4:1 GQA cuts KV memory bandwidth by 4x
    ffn_mult=8.0 / 3.0,          # Standard SwiGLU 2.67x ratio (avoids bloated MLP weights)
    sliding_window=512,          # Local attention chunks (linear complexity on 75% of layers)
    global_every=4,              # Global causal attention every 4th layer
    rope_base=12000.0,
    dropout=0.0,

    # Architectural stabilizers & modern upgrades
    use_qk_norm=True,            # Q/K RMSNorm prevents attention logit drift
    embed_scale=True,            # Gemma 2 embedding scaling stabilizes residual stream
    logit_soft_cap=30.0,         # Gemma 2 soft-capping bounds logits smoothly
    z_loss_weight=1e-4,          # PaLM/Gemma z-loss stabilizes logsumexp

    # Memory management
    grad_checkpoint=True,        # Retain only block boundaries in forward pass
    grad_checkpoint_every=1,
)

model = TensegrityLM(cfg)
model.to(DEVICE)

n_params = sum(p.numel() for p in model.parameters())
print(f"[TensegrityLM] configured model: {n_params:,} parameters ({n_params/1e6:.1f}M)")

optimizer = model.configure_optimizer(
    weight_decay=0.1,
    learning_rate=3e-4,
    betas=(0.9, 0.95),
    device_type=DEVICE_TYPE,
    use_muon=True,
    muon_lr=0.02,
    muon_momentum=0.95,
)
print(f"optimizer = {type(optimizer).__name__}")


[TensegrityLM] layers=4 n_embd=512 n_head=4 n_kv_head=2 vocab=259 block_size=256
[TensegrityLM] total parameters: 15,867,401 (15.87M)
param count = 15,867,401
optimizer = SingleDeviceMuonWithAuxAdam


## Training budget (single source of truth for the LR schedule)

In [10]:
# Training budget sized for 133M model on 1.75B tokens:
# At block_size=2048, micro BATCH_SIZE=16 consumes ~2.5 GB VRAM with grad_checkpointing.
# GRAD_ACCUM_STEPS=8 yields an effective batch size of 128 sequences (262,144 tokens/step).
TOTAL_TOKENS = 1_750_000_000
BATCH_SIZE = 16
GRAD_ACCUM_STEPS = 8
EFFECTIVE_BATCH_SIZE = BATCH_SIZE * GRAD_ACCUM_STEPS

tokens_per_step = EFFECTIVE_BATCH_SIZE * cfg.block_size
MAX_STEPS = TOTAL_TOKENS // tokens_per_step
WARMUP_STEPS = max(100, int(0.03 * MAX_STEPS))

# WSD (warmup-stable-decay) schedule
DECAY_STEPS = max(1, int(0.10 * MAX_STEPS))
STABLE_STEPS = max(1, MAX_STEPS - WARMUP_STEPS - DECAY_STEPS)

print(f"tokens/step   = {tokens_per_step:,}")
print(f"MAX_STEPS     = {MAX_STEPS:,}")
print(f"WARMUP_STEPS  = {WARMUP_STEPS:,}")
print(f"STABLE_STEPS  = {STABLE_STEPS:,}")
print(f"DECAY_STEPS   = {DECAY_STEPS:,}")
print(f"GRAD_ACCUM    = {GRAD_ACCUM_STEPS}")


tokens/step   = 131,072
MAX_STEPS     = 13,351
WARMUP_STEPS  = 400
STABLE_STEPS  = 11,616
DECAY_STEPS   = 1,335
GRAD_ACCUM    = 4


In [ ]:
gc.collect();
torch.cuda.empty_cache(),

## Train from scratch


In [ ]:
import gc
torch.cuda.empty_cache()
gc.collect()

RUN_INITIAL = os.path.join(RUNS_DIR, "initial")

# Ensure model and optimizer are on device and ready
model = TensegrityLM(cfg)
model.to(DEVICE)

optimizer = model.configure_optimizer(
    weight_decay=0.1,
    learning_rate=3e-4,
    betas=(0.9, 0.95),
    device_type=DEVICE_TYPE,
    use_muon=True,
    muon_lr=0.02,
    muon_momentum=0.95,
)

# --- RUN THE TRAINING ---
step, best_val = run_training(
    model, optimizer, cfg, RUN_INITIAL, DEVICE,
    data_dirs=DATA_INITIAL,
    data_weights=[1.0],

    max_steps=MAX_STEPS,
    warmup_steps=WARMUP_STEPS,

    schedule="wsd",
    stable_steps=STABLE_STEPS,
    decay_steps=DECAY_STEPS,

    learning_rate=3e-4,
    min_lr=3e-5,

    batch_size=BATCH_SIZE,
    grad_accum_steps=GRAD_ACCUM_STEPS,

    log_interval=5,
    eval_interval=500,
    eval_iters=5
)

# Save snapshot and stop recording after training.
try:
    torch.cuda.memory._dump_snapshot("snapshot.pickle")
except Exception as e:
    print(f"[warn] memory snapshot failed ({e}); training itself is unaffected, checkpoint is already saved")
torch.cuda.memory._record_memory_history(enabled=None)

print(f"finished initial training at step={step}, best_val_loss={best_val:.4f}")


[TensegrityLM] layers=4 n_embd=512 n_head=4 n_kv_head=2 vocab=259 block_size=256
[TensegrityLM] total parameters: 15,867,401 (15.87M)


In [ ]:
import torch
print(f"After model init: {torch.cuda.memory_allocated()/1e9:.2f} GB allocated, "
      f"{torch.cuda.memory_reserved()/1e9:.2f} GB reserved")

In [ ]:
import time

# Estimated throughput based on micro-batch and grad accum
# Each optimizer step processes (BATCH_SIZE * GRAD_ACCUM_STEPS * cfg.block_size) tokens
seconds_per_step = 0.8  # Adjust based on observed step time in train_log.jsonl
tokens_per_step = BATCH_SIZE * GRAD_ACCUM_STEPS * cfg.block_size
tokens_per_sec = tokens_per_step / seconds_per_step

remaining_steps = MAX_STEPS - step if 'step' in locals() else MAX_STEPS
remaining_time_hrs = (remaining_steps * seconds_per_step) / 3600

print(f"Throughput: {tokens_per_sec:,.0f} tokens/sec")
print(f"Remaining steps: {remaining_steps:,}")
print(f"Estimated time to completion: {remaining_time_hrs:.2f} hours")


## Load a checkpoint and generate


In [ ]:
import os
import torch
from tensegrity_lm import load_checkpoint, TensegrityConfig, TensegrityLM, ByteTokenizer

# Fallbacks in case earlier setup cells were skipped
if 'DEVICE' not in globals():
    from tensegrity_lm import pick_device
    DEVICE = pick_device("auto")
if 'tok' not in globals():
    tok = ByteTokenizer()

ckpt = load_checkpoint(os.path.join("/content/drive/MyDrive/tensegrity/nsfw001-001/initial/ckpt_latest.pt"), DEVICE)
loaded_cfg = TensegrityConfig(**ckpt["config"])

model = TensegrityLM(loaded_cfg)
model.load_state_dict(ckpt["model_state_dict"])
model.to(DEVICE)
cfg = loaded_cfg  # keep cfg in sync with the model from here on

print(f"loaded checkpoint from step={ckpt['step']}, best_val_loss={ckpt['best_val_loss']:.4f}")

model.eval()
prompt_ids = [ByteTokenizer.BOS] + tok.encode("Hello")
idx = torch.tensor([prompt_ids], dtype=torch.long, device=DEVICE)
with torch.no_grad():
    out = model.generate(idx, max_new_tokens=512, temperature=0.5, top_k=40, use_confidence_temp=True)
print(tok.decode(out[0].tolist()))

## Load new data (.parquet)

In [ ]:
import pandas as pd
import glob
import os

# Find parquet files in the data directory
data_path = "/content/data"
parquet_files = glob.glob(os.path.join(data_path, "*.parquet"))

for p_file in parquet_files:
    print(f"Reading {p_file}...")
    df = pd.read_parquet(p_file)

    # The text data seems to be in the 'content' column
    target_col = 'content'
    if target_col in df.columns:
        out_path = p_file.replace('.parquet', '.txt')
        print(f"Extracting text from '{target_col}' and saving to {out_path}...")

        # Join all text rows with double newlines and save
        text_data = "\n\n".join(df[target_col].astype(str).tolist())
        with open(out_path, 'w', encoding='utf-8') as f:
            f.write(text_data)

        # Remove the parquet file so `prepare_data` doesn't read its raw compressed bytes
        os.remove(p_file)
        print("Done. Original parquet file removed.")
    else:
        print(f"Could not find a '{target_col}' column in {p_file}. Available columns: {df.columns.tolist()}")

## Load new data (.jsonl)

In [ ]:
import json
import glob
import os

# Find jsonl files in the data directory
data_path = "/content/data"
jsonl_files = glob.glob(os.path.join(data_path, "*.jsonl"))

for j_file in jsonl_files:
    print(f"Reading {j_file}...")
    text_data = []

    with open(j_file, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
                # Attempt to extract text from common keys ('content' or 'text')
                if 'content' in obj and obj['content']:
                    text_data.append(str(obj['content']))
                elif 'text' in obj and obj['text']:
                    text_data.append(str(obj['text']))
            except json.JSONDecodeError:
                pass

    if text_data:
        out_path = j_file.replace('.jsonl', '.txt')
        print(f"Extracting {len(text_data)} lines of text and saving to {out_path}...")

        with open(out_path, 'w', encoding='utf-8') as out_f:
            out_f.write("\n\n".join(text_data))

        # Remove the jsonl file so it's not processed twice by prepare_data
        os.remove(j_file)
        print("Done. Original jsonl file removed.")
    else:
        print(f"Could not find valid text data in {j_file}.")

In [ ]:
import os

# Re-prepare data in case new files were added to the data directory
data_path = "/content/data"
DATA_INITIAL = os.path.join(data_path, "initial")
print(f"Re-preparing data from {data_path}...")
prepare_data(data_path, DATA_INITIAL, val_fraction=0.1, seed=SEED)
print("Data preparation complete.")

## Grow Model

In [ ]:
import dataclasses
from tensegrity_nb_lib import transplant_block
from tensegrity_lm import ffn_hidden_dim, TensegrityConfig

_GROWTH_LOCKED_FIELDS = [
    "vocab_size", "block_size", "n_embd", "n_head", "n_kv_head",
    "sliding_window", "global_every",
    "bias", "dropout", "init_std", "homeo_target", "homeo_weight",
    "use_confidence_head", "confidence_weight",
]

@torch.no_grad()
def grow_model(old_model, old_cfg, new_n_layer=None, ffn_growth_mult=None,
               new_n_embd=None, new_n_head=None, new_n_kv_head=None, device=None):
    """
    Create a strictly more capable model that, at the moment of growth,
    computes EXACTLY the same function as old_model (verified by the equality
    check in check_function_preserved).

    Three independent growth axes are supported, and any combination may be used:

      - new_n_layer:    append (new_n_layer - old_cfg.n_layer) new transformer
                        blocks at the end of the stack. New blocks have their
                        residual gains initialized to exactly 0, so they are
                        a no-op at init: x = x + 0*attn_out = x.
      - ffn_growth_mult: widen every block's FFN hidden dimension by this
                        factor (Net2Net "wider net"). New hidden units
                        contribute exactly 0 to the residual stream at init
                        because w_down's new input columns are zeroed.
      - new_n_embd / new_n_head: widen the residual stream dimension (n_embd).
                        Embedding columns, RMSNorm weights, and attention projections
                        are widened. By default, head_dim is kept fixed and n_head
                        (and n_kv_head proportionally for GQA) grows with n_embd,
                        preserving RoPE position frequencies and attention distribution.

    vocab_size, block_size, sliding_window, global_every, bias, and
    homeostasis/confidence hyperparameters are NOT changed by this function.
    """
    if device is None:
        device = next(old_model.parameters()).device

    new_cfg_kwargs = dataclasses.asdict(old_cfg)

    if new_n_layer is not None:
        assert new_n_layer >= old_cfg.n_layer, "new_n_layer must be >= old_cfg.n_layer"
        new_cfg_kwargs["n_layer"] = new_n_layer
        if new_n_layer != old_cfg.n_layer:
            # let TensegrityConfig.__post_init__ recompute gain_init for the new depth;
            # this only affects the *new* blocks (old blocks' gains are copied below).
            new_cfg_kwargs["gain_init"] = None

    if ffn_growth_mult is not None:
        assert ffn_growth_mult >= 1.0, "ffn_growth_mult must be >= 1.0"
        new_cfg_kwargs["ffn_mult"] = old_cfg.ffn_mult * ffn_growth_mult

    if new_n_embd is not None or new_n_head is not None:
        old_head_dim = old_cfg.n_embd // old_cfg.n_head
        if new_n_embd is not None:
            assert new_n_embd >= old_cfg.n_embd, "new_n_embd must be >= old_cfg.n_embd"
            new_cfg_kwargs["n_embd"] = new_n_embd
        else:
            assert new_n_head >= old_cfg.n_head, "new_n_head must be >= old_cfg.n_head"
            new_n_embd = new_n_head * old_head_dim
            new_cfg_kwargs["n_embd"] = new_n_embd

        # head_dim must stay fixed: precompute_rope's inv_freq is a function of head_dim
        # itself, so changing it would change the RoPE rotation angle applied to every
        # *existing* channel, not just add new ones -- there is no way to keep old
        # channels' attention scores exact under a head_dim change. Earlier revisions of
        # this function had a "custom head_dim" fallback for new_n_embd values that
        # don't divide old_head_dim evenly; it looked like it worked (fresh random-init
        # models pass loose atol checks) but was never actually exact -- verified to
        # diverge by ~0.08 max logit difference on a realistically-trained model, well
        # outside any reasonable tolerance. Only the fixed-head_dim convention is exact.
        if new_n_embd % old_head_dim != 0:
            raise ValueError(
                f"new_n_embd ({new_n_embd}) is not a multiple of the existing head_dim "
                f"({old_head_dim} = old n_embd {old_cfg.n_embd} // old n_head {old_cfg.n_head}). "
                f"grow_model only supports function-preserving growth that keeps head_dim "
                f"fixed (adding more heads, not widening each head) -- pick a new_n_embd "
                f"that's a multiple of {old_head_dim}, e.g. "
                f"{old_head_dim * (new_n_embd // old_head_dim + 1)}."
            )
        implied_n_head = new_n_embd // old_head_dim
        if new_n_head is None:
            new_n_head = implied_n_head
        elif new_n_head != implied_n_head:
            raise ValueError(
                f"new_n_head={new_n_head} would change head_dim from {old_head_dim} to "
                f"{new_n_embd // new_n_head} (not function-preserving under RoPE -- see "
                f"above). For new_n_embd={new_n_embd}, the only head_dim-preserving choice "
                f"is new_n_head={implied_n_head}; omit new_n_head to have it inferred."
            )
        assert new_n_head >= old_cfg.n_head, "new_n_head must be >= old_cfg.n_head"
        new_cfg_kwargs["n_head"] = new_n_head

        if new_n_kv_head is None:
            gqa_ratio = old_cfg.n_head // old_cfg.n_kv_head
            new_n_kv_head = max(1, new_n_head // gqa_ratio)
        new_cfg_kwargs["n_kv_head"] = new_n_kv_head

    new_cfg = TensegrityConfig(**new_cfg_kwargs)

    # Sanity: confirm none of the "locked" fields drifted.
    locked_fields = list(_GROWTH_LOCKED_FIELDS)
    if new_n_embd is not None or new_n_head is not None:
        for f in ["n_embd", "n_head", "n_kv_head"]:
            if f in locked_fields:
                locked_fields.remove(f)

    for field_name in locked_fields:
        old_val = getattr(old_cfg, field_name)
        new_val = getattr(new_cfg, field_name)
        assert old_val == new_val, (
            f"grow_model must not change '{field_name}' "
            f"(old={old_val!r}, new={new_val!r}); this would break function preservation."
        )

    old_hidden = ffn_hidden_dim(old_cfg.n_embd, old_cfg.ffn_mult, old_cfg.ffn_align)
    new_hidden = ffn_hidden_dim(new_cfg.n_embd, new_cfg.ffn_mult, new_cfg.ffn_align)
    if new_hidden < old_hidden:
        raise ValueError(
            f"Resulting ffn hidden_dim ({new_hidden}) is smaller than the current one "
            f"({old_hidden}). Increase ffn_growth_mult or leave it as None."
        )

    new_model = TensegrityLM(new_cfg)
    new_model.to(device)
    old_model.to(device)

    # --- Global / shared modules ---
    if old_cfg.n_embd != new_cfg.n_embd:
        scale = (old_cfg.n_embd / new_cfg.n_embd) ** 0.5
        new_model.token_emb.weight.data.zero_()
        new_model.token_emb.weight.data[:, :old_cfg.n_embd].copy_(old_model.token_emb.weight.data)
        new_model.ln_f.weight.data[:old_cfg.n_embd].copy_(old_model.ln_f.weight.data * scale)
        # See transplant_block's matching comment: eps must scale by old_D/new_D too,
        # not just the weight vector, for RMSNorm growth to be exact rather than approximate.
        new_model.ln_f.eps = old_model.ln_f.eps * (scale ** 2)
        if old_cfg.use_confidence_head and new_cfg.use_confidence_head:
            new_model.confidence_head.weight.data.zero_()
            new_model.confidence_head.weight.data[:, :old_cfg.n_embd].copy_(old_model.confidence_head.weight.data)
            new_model.confidence_head.bias.data.copy_(old_model.confidence_head.bias.data)
    else:
        new_model.token_emb.weight.data.copy_(old_model.token_emb.weight.data)  # also lm_head (tied)
        new_model.ln_f.weight.data.copy_(old_model.ln_f.weight.data)
        if old_cfg.use_confidence_head and new_cfg.use_confidence_head:
            new_model.confidence_head.weight.data.copy_(old_model.confidence_head.weight.data)
            new_model.confidence_head.bias.data.copy_(old_model.confidence_head.bias.data)

    # --- Existing blocks: copy / widen ---
    for i in range(old_cfg.n_layer):
        transplant_block(old_model.blocks[i], new_model.blocks[i], old_cfg, new_cfg)

    # --- New blocks (depth growth only): zero residual gains -> exact no-op ---
    for i in range(old_cfg.n_layer, new_cfg.n_layer):
        new_model.blocks[i].attn_gain.data.zero_()
        new_model.blocks[i].ffn_gain.data.zero_()

    return new_model, new_cfg

In [ ]:
import os
import torch
from tensegrity_lm import load_checkpoint, TensegrityConfig, TensegrityLM, save_checkpoint

# 1. Load the existing checkpoint
ckpt_path = "/content/drive/MyDrive/tensegrity/npc001/initial/ckpt_latest.pt"
ckpt = load_checkpoint(ckpt_path, DEVICE)
old_cfg = TensegrityConfig(**ckpt["config"])

old_model = TensegrityLM(old_cfg)
old_model.load_state_dict(ckpt["model_state_dict"])
old_model.to(DEVICE)

print(f"Original model parameters: {sum(p.numel() for p in old_model.parameters()):,} | Layers: {old_cfg.n_layer}")

# 2. Grow the model across all supported axes (depth, width, and FFN expansion)
new_n_layer = old_cfg.n_layer + 2  # Add 2 more transformer blocks (reduced for ~50M size)
ffn_growth_mult = 1.25             # Expand FFN hidden dimension by 1.25x

# Head dim is old_cfg.n_embd // old_cfg.n_head (e.g. 512 // 16 = 32)
# new_n_embd must be a multiple of head_dim (32). Let's increase it by 128.
new_n_embd = old_cfg.n_embd + 128  # e.g., 512 + 128 = 640
new_n_head = old_cfg.n_head + 4    # e.g., 16 + 4 = 20 (640 / 32 = 20)

print(f"Growing model to {new_n_layer} layers, {new_n_embd} embedding size, and {ffn_growth_mult}x FFN...")
new_model, new_cfg = grow_model(
    old_model=old_model,
    old_cfg=old_cfg,
    new_n_layer=new_n_layer,
    ffn_growth_mult=ffn_growth_mult,
    new_n_embd=new_n_embd,
    new_n_head=new_n_head,
    device=DEVICE
)

print(f"New model parameters: {sum(p.numel() for p in new_model.parameters()):,} | Layers: {new_cfg.n_layer}")

# 3. Save the new grown checkpoint
grown_ckpt_path = "/content/runs/nfsw-v0.1.0-grown.pt"

# Use configure_optimizer to ensure the saved optimizer state has the correct parameter groups
dummy_optimizer = new_model.configure_optimizer(
    weight_decay=0.1,
    learning_rate=3e-4,
    betas=(0.9, 0.95),
    device_type=DEVICE_TYPE
)

save_checkpoint(
    grown_ckpt_path,
    new_model,
    optimizer=dummy_optimizer,
    cfg=new_cfg,
    step=ckpt["step"],
    best_val_loss=ckpt["best_val_loss"],
    rng_state=ckpt.get("rng_state")
)
print(f"Successfully saved grown model to {grown_ckpt_path}")


## Resume Training from Checkpoint

In [ ]:
import tensegrity_lm
import tensegrity_nb_lib
import torch
import numpy as np

# Patch restore_rng_state to fix the torch.ByteTensor device error when loading on GPU
def patched_restore_rng_state(state):
    if state is None:
        return

    if "torch" in state and isinstance(state["torch"], torch.Tensor):
        # Ensure the tensor is on the CPU before setting the state
        state["torch"] = state["torch"].cpu()
        torch.set_rng_state(state["torch"])

    if "numpy" in state:
        np.random.set_state(state["numpy"])

    if "cuda" in state and torch.cuda.is_available():
        if isinstance(state["cuda"], list):
            # Ensure all CUDA RNG states are on the CPU
            state["cuda"] = [s.cpu() if isinstance(s, torch.Tensor) else s for s in state["cuda"]]
        try:
            torch.cuda.set_rng_state_all(state["cuda"])
        except Exception as e:
            print(f"[warn] failed to restore CUDA RNG state ({e})")

# Apply the patch to both modules
tensegrity_lm.restore_rng_state = patched_restore_rng_state
tensegrity_nb_lib.restore_rng_state = patched_restore_rng_state

print("Patched restore_rng_state successfully!")

In [ ]:
import shutil
import os

# We will resume the training in a new directory for the base model
RUN_RESUME = os.path.join(RUNS_DIR, "continued_base")
os.makedirs(RUN_RESUME, exist_ok=True)

# Point to your base model checkpoint (fixed typo from nfsw to nsfw)
base_ckpt_path = "/content/drive/MyDrive/tensegrity/npc001/initial/ckpt_latest.pt"

# If you meant to use the v0.1.4 model from earlier, uncomment the next line:
# base_ckpt_path = "/content/drive/MyDrive/custom_model/nsfw/runs/nsfw-v0.1.4.pt"

# Copy the base checkpoint to act as the latest checkpoint in the new run directory
shutil.copy2(base_ckpt_path, os.path.join(RUN_RESUME, "ckpt_latest.pt"))

# 1. Read the config from the checkpoint to ensure architecture matches
ckpt_resume = load_checkpoint(base_ckpt_path, DEVICE)
cfg_resume = TensegrityConfig(**ckpt_resume["config"])

# 2. Build the model and optimizer
model_resume = TensegrityLM(cfg_resume)
model_resume.to(DEVICE)

optimizer_resume = model_resume.configure_optimizer(
    weight_decay=0.1,
    learning_rate=3e-4,
    betas=(0.9, 0.95),
    device_type=DEVICE_TYPE,
    use_muon=True, # Added to match the original optimizer configuration
    muon_lr=0.02,  # Added to match the original optimizer configuration
    muon_momentum=0.95, # Added to match the original optimizer configuration
)

# 3. Run training with init_from="resume"
step_resume, best_val_resume = run_training(
    model=model_resume,
    optimizer=optimizer_resume,
    cfg=cfg_resume,
    out_dir=RUN_RESUME,
    device=DEVICE,
    data_dirs=[DATA_INITIAL],

    max_steps=MAX_STEPS,
    warmup_steps=WARMUP_STEPS,
    learning_rate=3e-4,
    min_lr=3e-5,
    batch_size=BATCH_SIZE,

    log_interval=50,
    eval_interval=250,
    eval_iters=10,

    init_from="resume"
)

print(f"Finished resumed base model training at step={step_resume}, best_val_loss={best_val_resume:.4f}")

## Compute Fisher Information (EWC)
Before moving to the new dataset, we compute the Fisher diagonal on the old dataset. This identifies which weights are most important for the knowledge we want to retain.

In [ ]:
import gc
import torch
from tensegrity_nb_lib import compute_fisher_diagonal

# Free up memory before computing Fisher
torch.cuda.empty_cache()
gc.collect()

print("Computing Fisher information matrix diagonal...")
ewc_fisher, ewc_old_params = compute_fisher_diagonal(
    model="/content/drive/MyDrive/Colab Notebooks/models/run7/initial/ckpt_latest.pt",
    cfg=cfg,
    data_dir=DATA_INITIAL,  # The dataset we just trained on
    device=DEVICE,
    device_type=DEVICE_TYPE,
    n_batches=50,           # Standard number of batches for estimation
    batch_size=BATCH_SIZE
)
print("Fisher computation complete! You can now pass these to `run_training` with `ewc_lambda > 0` for your new dataset.")

## Fine-tune with EWC
Now we train on the new dataset while applying the EWC penalty to protect important weights from the previous dataset.

In [ ]:
import os
import shutil

# Create a new directory for the EWC fine-tuning phase
RUN_EWC = os.path.join(RUNS_DIR, "ewc_finetune")
os.makedirs(RUN_EWC, exist_ok=True)

# Copy the latest checkpoint from the previous run so we can cleanly resume
shutil.copy2(os.path.join(RUN_RESUME, "ckpt_latest.pt"), os.path.join(RUN_EWC, "ckpt_latest.pt"))

# EWC Lambda controls the penalty strength.
# Lowered to 10.0 to reduce friction and stabilize the loss.
EWC_LAMBDA = 10.0

print(f"Starting fine-tuning with EWC (lambda={EWC_LAMBDA})...")
step_ewc, best_val_ewc = run_training(
    model=model_resume,
    optimizer=optimizer_resume,
    cfg=cfg_resume,
    out_dir=RUN_EWC,
    device=DEVICE,
    data_dirs=[DATA_INITIAL],  # <-- Ensure this points to your new dataset

    max_steps=MAX_STEPS,
    warmup_steps=WARMUP_STEPS,
    learning_rate=3e-5,        # Lowered learning rate to stabilize fine-tuning
    min_lr=3e-6,
    batch_size=BATCH_SIZE,

    log_interval=50,
    eval_interval=250,
    eval_iters=10,

    init_from="resume",

    # Pass the Fisher information and old parameters to enable EWC
    ewc_fisher=ewc_fisher,
    ewc_old_params=ewc_old_params,
    ewc_lambda=EWC_LAMBDA
)

print(f"Finished EWC fine-tuning at step={step_ewc}, best_val_loss={best_val_ewc:.4f}")

## Export to safetensors


In [ ]:
EXPORT_DIR = os.path.join(RUNS_DIR, "export")
EXPORT_NAME = "tensegrity"

weights_path, config_path = export_to_safetensors(model_resume, cfg_resume, EXPORT_DIR, name=EXPORT_NAME)
print(f"weights : {weights_path}")
print(f"config  : {config_path}")

## Reload from safetensors and verify


In [ ]:
reloaded_model, reloaded_cfg = load_from_safetensors(weights_path, config_path, device=DEVICE)
reloaded_model.eval()
model_resume.eval()

torch.manual_seed(SEED)
x_check = torch.randint(0, cfg_resume.vocab_size, (1, 16), device=DEVICE)
with torch.no_grad():
    logits_a, _, _ = model_resume(x_check, targets=None)
    logits_b, _, _ = reloaded_model(x_check, targets=None)
print(f"max logit difference after safetensors roundtrip: {(logits_a - logits_b).abs().max().item():.3e}")

prompt_ids = [ByteTokenizer.BOS] + tok.encode("You ")
idx = torch.tensor([prompt_ids], dtype=torch.long, device=DEVICE)
with torch.no_grad():
    out = reloaded_model.generate(idx, max_new_tokens=120, temperature=0.9, top_k=40, use_confidence_temp=True)
print(tok.decode(out[0].tolist()))

## PyTorch Native INT8 Quantization (Dynamic)
Because `TensegrityLM` is a custom architecture, it cannot be directly converted to GGUF without writing a custom C++ backend for `llama.cpp`. Instead, we can use PyTorch's built-in dynamic quantization to convert the linear layers to 8-bit integers.

In [ ]:
import torch
import os

print("Applying PyTorch dynamic INT8 quantization...")

# Move model to CPU for dynamic quantization
model_cpu = model_resume.to('cpu')

# Quantize all nn.Linear layers to INT8 (qint8)
quantized_model = torch.ao.quantization.quantize_dynamic(
    model_cpu,
    {torch.nn.Linear},
    dtype=torch.qint8
)

# Save the PyTorch quantized model
# Note: Safetensors doesn't easily support PyTorch's custom qint8 tensor format,
# so we use standard torch.save for the quantized checkpoint.
quantized_path = os.path.join(EXPORT_DIR, "tensegrity_int8_quantized.pt")
torch.save(quantized_model.state_dict(), quantized_path)

# Compare file sizes
original_size = os.path.getsize(weights_path) / (1024 * 1024)
quant_size = os.path.getsize(quantized_path) / (1024 * 1024)

print(f"\nOriginal Safetensors Size: {original_size:.2f} MB")
print(f"Quantized INT8 Size:       {quant_size:.2f} MB")
print(f"Reduction:                 {(1 - quant_size/original_size)*100:.1f}%")

# Test the quantized model
print("\nTesting quantized model generation:")
prompt_ids = [ByteTokenizer.BOS] + tok.encode("You ")
idx = torch.tensor([prompt_ids], dtype=torch.long, device='cpu')

with torch.no_grad():
    out = quantized_model.generate(idx, max_new_tokens=50, temperature=0.8)

print("\nOutput:")
print(tok.decode(out[0].tolist()))

# Move the original model back to GPU if needed later
model_resume.to(DEVICE)
print("\nOriginal model returned to device.")

## Download Checkpoint Programmatically
Run this cell with `Shift + Enter` to download the latest checkpoint without needing a mouse.

In [ ]:
try:
    from google.colab import files
    import os

    latest_ckpt_path = os.path.join(RUNS_DIR, "initial", "ckpt_latest.pt")

    if os.path.exists(latest_ckpt_path):
        print(f"Triggering browser download for {latest_ckpt_path}...")
        files.download(latest_ckpt_path)
    else:
        print(f"File not found at: {latest_ckpt_path}")
except (ImportError, ModuleNotFoundError):
    print("google.colab not available; checkpoints are saved locally in RUNS_DIR.")


## Resume from Coherent Checkpoint (Lower LR)

In [ ]:
import os
import shutil

# 1. Create a fresh directory for this new attempt
RUN_FRESH = os.path.join(RUNS_DIR, "fresh_start")
os.makedirs(RUN_FRESH, exist_ok=True)

coherent_ckpt_path = "/content/drive/MyDrive/custom_model/nsfw/runs/nsfw-v0.1.4.pt"
latest_ckpt_path = os.path.join(RUN_FRESH, "ckpt_latest.pt")

# 2. Copy the coherent checkpoint to act as our starting point
shutil.copy2(coherent_ckpt_path, latest_ckpt_path)
print(f"Copied coherent checkpoint to {RUN_FRESH}")

# 3. Read config and build model/optimizer
ckpt_fresh = load_checkpoint(latest_ckpt_path, DEVICE)
cfg_fresh = TensegrityConfig(**ckpt_fresh["config"])

model_fresh = TensegrityLM(cfg_fresh)
model_fresh.to(DEVICE)

# Note the lowered learning rate here!
optimizer_fresh = model_fresh.configure_optimizer(
    weight_decay=0.1,
    learning_rate=3e-5,
    betas=(0.9, 0.95),
    device_type=DEVICE_TYPE,
)

# 4. Resume training gently
step_fresh, best_val_fresh = run_training(
    model=model_fresh,
    optimizer=optimizer_fresh,
    cfg=cfg_fresh,
    out_dir=RUN_FRESH,
    device=DEVICE,
    data_dirs=[DATA_INITIAL],

    max_steps=MAX_STEPS,
    warmup_steps=WARMUP_STEPS,
    # Much lower learning rate to preserve coherence:
    learning_rate=3e-5,
    min_lr=3e-6,
    batch_size=BATCH_SIZE,

    log_interval=50,
    eval_interval=250,
    eval_iters=10,

    init_from="resume"
)

print(f"Finished gentle training at step={step_fresh}, best_val_loss={best_val_fresh:.4f}")